# Football Tracking -- Player grouping & naming (notebook 2 of 2)
Input: the tracking notebook's outputs (`..._relinked_forced.pkl` + `grouping_inputs.pkl`) and your reviewed
`union_groups.csv`. Output: a cache where every player has **one ID = team offset + shirt number** (BAR = 100+,
ATM = 200+: `102` Cubarsi, `202` Gimenez, `207` Griezmann) plus `name`, `number`, `team`, `label` on every box;
officials keep IDs 1-5. Also a Parquet table (one row per box) for the analytics notebooks.

**MODE = "apply"** (default): Cells 1-2, then 10, 10c, 10e, 10f, 11-12. Uses your reviewed CSV as-is; no review work.
**MODE = "review"** (set now): Cells 1-2, 3-9, then 10, 10c, 10e, 10f, 11-12. Cells 3-9 rebuild tiles, proposals and the review steps (your decisions are already typed in
Cells 5 and 9). Group numbers and piece ids only match your decisions if the tracking notebook produced exactly the
same pieces -- Cell 2 checks this and warns.

**Keep `union_groups.csv` safe**: add `/kaggle/working/union/union_groups.csv` from the session where you did the review
to a dataset -- it is the only record of your review.

**Cell 10c (gap recovery)** puts back boxes the zero-switch tracking deleted (contacts, short pieces, ghost IDs, unassigned pieces) when the same raw Deep-EIoU track is the same player on both sides, then interpolates gaps <= 2 s. No switch is created and the tracking notebook does **not** need to be re-run.

**Cell 10e (keep every box)**: no detection box is ever removed from the final output -- boxes without a confirmed name are kept as `unknown` / `untracked`.

**Cell 10f (resolve unknowns)**: every unknown box becomes one of the 25 people (auto, with a score) or, when it cannot be any of them, `duplicate` / `off-pitch` / `extra` -- no unknown left, no box removed.

## Cell 1 -- Config: inputs, players (group -> name, number, team), id scheme

In [ ]:
import os, math, pickle, shutil, bisect
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter, deque
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

# "apply"  : use your REVIEWED union CSV -> final cache with id = team offset + shirt number + name (fast)
# "review" : build group proposals from scratch and review them (Cells 4-9), then apply
MODE = "review"

# ---- inputs (outputs of the tracking notebook + your reviewed CSV) ----
TRACKING_CACHE_PATH  = "/kaggle/input/<your-dataset>/barca_atletico_tracking_cache_deep_eiou_relinked_forced.pkl"
GROUPING_INPUTS_PATH = "/kaggle/input/<your-dataset>/grouping_inputs.pkl"
REVIEWED_UNION_CSV   = "/kaggle/input/<your-dataset>/union_groups.csv"     # your reviewed CSV (MODE="apply")
VIDEO_PATH           = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv"
# ---- Cell 10c (gap recovery) inputs ----
RAW_TRACKS_PATH      = "/kaggle/input/deep-eiou-full-match/deep_eiou_tracks.parquet"      # raw Deep-EIoU tracks (same file as notebook 1)
HOMOGRAPHY_PATH      = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/homography_cache_barca_atletico_firsthalf.pkl"
SHOT_CACHE_PATH      = "/kaggle/input/<your-dataset>/shot_boundaries.npz"   # notebook 1 Cell 7b output, if you saved it
COMPUTE_SHOTS_IF_MISSING = True                                              # no shot cache -> compute it from the video (~5-10 min)
DETECTION_CACHE_PATH = "/kaggle/input/<your-dataset>/<rf-detr detection cache>.pkl"  # Cell 10e: keep every detection (None = raw tracker boxes only)
RAW_EMBS_PATH        = "/kaggle/input/deep-eiou-full-match/deep_eiou_track_embeddings.npz"   # Cell 10f: appearance of every raw box

# ---- outputs ----
WORK_DIR         = "/kaggle/working/union"
UNION_CSV        = os.path.join(WORK_DIR, "union_groups.csv")                # working copy (edited by the review cells)
FINAL_CACHE_PATH = "/kaggle/working/barca_atletico_tracking_cache_named.pkl"
FINAL_TABLE_PATH = "/kaggle/working/barca_atletico_tracks_named.parquet"

# ---- who is who: group number (from your review) -> (name, shirt number, team) ----
PLAYERS = {
    1: ("Griezmann", 7, "ATM"),    2: ("Fermin", 16, "BAR"),      3: ("Alvarez", 19, "ATM"),
    4: ("Lewandowski", 9, "BAR"),  5: ("Gavi", 6, "BAR"),         6: ("Inigo", 5, "BAR"),
    7: ("Pedri", 8, "BAR"),        8: ("Cubarsi", 2, "BAR"),      9: ("Balde", 3, "BAR"),
    10: ("Casado", 17, "BAR"),     11: ("Llorente", 14, "ATM"),   12: ("De Paul", 5, "ATM"),   # number 5 not read on video -- check
    13: ("Raphinha", 11, "BAR"),   14: ("Lenglet", 15, "ATM"),    15: ("Gallagher", 4, "ATM"),
    16: ("Barrios", 8, "ATM"),     17: ("Javi Galan", 21, "ATM"), 18: ("Kounde", 23, "BAR"),
    19: ("Simeone", 22, "ATM"),    20: ("Gimenez", 2, "ATM"),
}
# shirt numbers repeat across teams (Cubarsi 2 / Gimenez 2) -> id = team offset + number: 102 Cubarsi, 202 Gimenez
TEAM_ID_OFFSET = {"BAR": 100, "ATM": 200}
# forced officials from the tracking notebook keep ids 1-5 (edit names / teams if you know them)
OFFICIALS = {1: ("Pena", 13, "BAR"), 2: ("Oblak", 13, "ATM"),
             3: ("Referee", None, None), 4: ("Assistant referee y0", None, None), 5: ("Assistant referee y68", None, None)}
KEEP_UNASSIGNED = False        # True = keep pieces you did not assign, as ids 1000+ (name "unassigned")
UNASSIGNED_ID_START = 1000

GROUP_NAMES = {g: f"{n} {k}" for g, (n, k, t) in PLAYERS.items()}
UNION_DIR = WORK_DIR
os.makedirs(WORK_DIR, exist_ok=True)
_ids = [TEAM_ID_OFFSET[t] + k for (n, k, t) in PLAYERS.values()]
assert len(_ids) == len(set(_ids)), "two players map to the same id -- check numbers/teams"
print(f"MODE = {MODE} | players: {len(PLAYERS)} "
      f"({Counter(t for _, _, t in PLAYERS.values())}) | final ids {sorted(_ids)}")


## Cell 2 -- Load tracking outputs, per-ID arrays, check the CSV belongs to this tracking run

In [ ]:
with open(TRACKING_CACHE_PATH, "rb") as fh:
    _d = pickle.load(fh)
final_tracking_cache, locked_class_by_id = _d["tracking_cache"], _d["locked_class_by_id"]
with open(GROUPING_INPUTS_PATH, "rb") as fh:
    gi = pickle.load(fh)
for _k, _v in gi["params"].items():
    globals()[_k] = _v                     # FPS, n_frames, MAX_SPEED_MPS, POS_SLACK_M, MERGE_OVERLAP_TOL, ...
TEAM_CENTROIDS, id_feat = gi["TEAM_CENTROIDS"], gi["id_feat"]
print(f"Cache: {len(final_tracking_cache)} frames | player IDs with features: {len(id_feat)} | "
      f"FPS {FPS}, frames {n_frames}")


def _norm(v):
    n = float(np.linalg.norm(v))
    return v / n if n > 1e-12 else v


def reachable(a, b):
    """True if a person seen at anchor a=(frame, pos_m) can be at anchor b. Unknown position -> True."""
    if a is None or b is None:
        return True
    dt = max(abs(b[0] - a[0]), 1) / FPS
    return float(np.linalg.norm(np.asarray(b[1]) - np.asarray(a[1]))) <= MAX_SPEED_MPS * dt + POS_SLACK_M


def team_label(emb):
    if TEAM_CENTROIDS is None or emb is None:
        return -1
    s = TEAM_CENTROIDS @ np.asarray(emb, dtype=np.float32)
    return int(np.argmax(s)) if abs(float(s[0] - s[1])) >= TEAM_MIN_MARGIN else -1


def _feasible(sa, sb):
    s = sorted(sa + sb, key=lambda x: x[0])
    return all(reachable(p[3], q[2]) for p, q in zip(s, s[1:]))


# ---- per player-ID arrays ----
u_obs = defaultdict(list)                                   # id -> [(frame, bbox)]
for f, d in final_tracking_cache.items():
    for t in d["tracks"]:
        if locked_class_by_id.get(t["track_id"]) == "player":
            u_obs[t["track_id"]].append((f, t["bbox"]))
u_ids = sorted(u_obs)
n_u = len(u_ids)
u_row = {c: r for r, c in enumerate(u_ids)}
U_E = np.zeros((n_u, 512), dtype=np.float32)
u_has = np.zeros(n_u, dtype=bool)
u_segs, rows_, cols_ = [], [], []
for r, cid in enumerate(u_ids):
    ft = id_feat.get(cid)
    if ft is not None and ft["emb_sum"] is not None:
        U_E[r] = ft["emb_sum"]; u_has[r] = True
    u_segs.append(list(ft["segs"]) if ft is not None else [])
    fr = np.unique([f for f, _ in u_obs[cid]])
    rows_.append(np.full(len(fr), r)); cols_.append(fr)
u_nfr = np.array([len(u_obs[c]) for c in u_ids])
U_P = np.stack([_norm(e) if h else e for e, h in zip(U_E, u_has)]).astype(np.float32)
u_team = np.array([team_label(U_P[r]) if u_has[r] else -1 for r in range(n_u)])
U_Occ = csr_matrix((np.ones(sum(len(c) for c in cols_), dtype=np.float32),
                    (np.concatenate(rows_), np.concatenate(cols_))), shape=(n_u, n_frames + 1))
U_OV = np.rint((U_Occ @ U_Occ.T).toarray()).astype(np.int64)
np.fill_diagonal(U_OV, 0)
print(f"Player IDs: {n_u} | team labels: {dict(Counter(u_team.tolist()))} (-1 = unsure)")

# ---- the reviewed CSV must belong to THIS tracking run ----
if MODE == "apply":
    assert os.path.exists(REVIEWED_UNION_CSV), f"reviewed CSV not found: {REVIEWED_UNION_CSV}"
    shutil.copy(REVIEWED_UNION_CSV, UNION_CSV)
if os.path.exists(UNION_CSV):
    _chk = pd.read_csv(UNION_CSV)
    _missing = set(_chk.id.astype(int)) - set(u_ids)
    _extra = set(u_ids) - set(_chk.id.astype(int))
    _ndet = dict(zip(_chk.id.astype(int), _chk.n_det))
    _diff = sum(1 for c in u_ids if c in _ndet and _ndet[c] != len(u_obs[c]))
    print(f"CSV check: ids in CSV not in cache {len(_missing)} | ids in cache not in CSV {len(_extra)} | "
          f"detection-count mismatches {_diff}")
    if _missing or _extra or _diff:
        print("  [WARNING] the CSV was made on a DIFFERENT tracking run (ids changed). Group assignments would be "
              "wrong -> use the tracking cache of the run you reviewed, or MODE='review' to redo the grouping.")


## Cell 3 -- (review) One tile per player ID from the video

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # one tile per player ID (largest box in the middle half of the ID), sequential video read (review mode only)
    TILE_H, TILE_W, TILE_COLS = 128, 64, 16
    SHOW_GROUPS_INLINE = True
    font = cv2.FONT_HERSHEY_SIMPLEX
    tile, tile_frame = {}, {}


    def _letter(img):
        h, w = img.shape[:2]
        s = min(TILE_W / max(w, 1), TILE_H / max(h, 1))
        rimg = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
        out = np.full((TILE_H, TILE_W, 3), 40, np.uint8)
        y0, x0 = (TILE_H - rimg.shape[0]) // 2, (TILE_W - rimg.shape[1]) // 2
        out[y0:y0 + rimg.shape[0], x0:x0 + rimg.shape[1]] = rimg
        return out


    def _sheet(title, cids, sims, path, show):
        n = len(cids); rows = max(1, math.ceil(n / TILE_COLS)); LH, HH = 26, 32
        sh = np.full((HH + rows * (TILE_H + LH), TILE_COLS * TILE_W, 3), 255, np.uint8)
        cv2.putText(sh, title, (6, 22), font, 0.55, (0, 0, 0), 2, cv2.LINE_AA)
        for k, (cid, sm) in enumerate(zip(cids, sims)):
            r, c = divmod(k, TILE_COLS); y, x = HH + r * (TILE_H + LH), c * TILE_W
            if cid in tile:
                sh[y:y + TILE_H, x:x + TILE_W] = tile[cid]
            cv2.rectangle(sh, (x, y), (x + TILE_W - 1, y + TILE_H - 1), (200, 200, 200), 1)
            cv2.putText(sh, f"{cid}", (x + 2, y + TILE_H + 11), font, 0.36, (0, 0, 180), 1, cv2.LINE_AA)
            t = tile_frame.get(cid, 0) / FPS
            cv2.putText(sh, f"{int(t // 60):02d}:{int(t % 60):02d}" + (f" {sm:.2f}" if sm is not None and sm == sm else ""),
                        (x + 2, y + TILE_H + 22), font, 0.3, (80, 80, 80), 1, cv2.LINE_AA)
        cv2.imwrite(path, sh)
        if show:
            plt.figure(figsize=(TILE_COLS * 0.75, 0.4 + rows * 1.8))
            plt.imshow(cv2.cvtColor(sh, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()


    want = defaultdict(list)
    for cid in u_ids:
        o = sorted(u_obs[cid], key=lambda x: x[0])
        mid = o[len(o) // 4: max(len(o) // 4 + 1, 3 * len(o) // 4)]
        rep = max(mid, key=lambda x: x[1][3] - x[1][1])
        want[rep[0]].append((cid, rep[1]))
    cap = cv2.VideoCapture(VIDEO_PATH)
    assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
    last, f = max(want), 0
    with tqdm(total=last + 1, desc="Reading video (tiles)", unit="frame", mininterval=1.0) as pbar:
        while f <= last:
            if f in want:
                ok, img = cap.read()
                if not ok: break
                H, W = img.shape[:2]
                for cid, (x1, y1, x2, y2) in want[f]:
                    px, py = (x2 - x1) * 0.1, (y2 - y1) * 0.1
                    a, b = int(max(0, x1 - px)), int(max(0, y1 - py))
                    c, d = int(min(W, x2 + px)), int(min(H, y2 + py))
                    if c > a and d > b:
                        tile[cid] = _letter(img[b:d, a:c]); tile_frame[cid] = f
            elif not cap.grab():
                break
            f += 1; pbar.update(1)
    cap.release()
    print(f"Tiles: {len(tile)}")


## Cell 4 -- (review) Group proposals + one sheet per group

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # Proposals: one group per player (never co-visible, same team, reachable), low threshold on purpose.
    SUGGEST_SIM = 0.55
    UNASSIGNED_PER_PAGE = 96
    REBUILD_PROPOSALS = False      # False: keep an existing working CSV (continue a review); True: start over

    if os.path.exists(UNION_CSV) and not REBUILD_PROPOSALS:
        print(f"{UNION_CSV} exists -> kept (set REBUILD_PROPOSALS=True to start over)")
    else:
        # ---- 2. constrained agglomerative proposals ----
        def _feasible(sa, sb):
            s = sorted(sa + sb, key=lambda x: x[0])
            return all(reachable(p[3], q[2]) for p, q in zip(s, s[1:]))

        P, E, OV, tm = U_P.copy(), U_E.copy(), U_OV.copy(), u_team.copy()
        segs = [list(s) for s in u_segs]
        groups = [[r] for r in range(n_u)]
        alive = np.ones(n_u, dtype=bool)

        def _valid(a):
            v = alive & u_has & (OV[a] <= MERGE_OVERLAP_TOL)
            if tm[a] >= 0:
                v &= (tm == tm[a]) | (tm < 0)
            v[a] = False
            return v

        M = np.full((n_u, n_u), -np.inf, dtype=np.float32)
        S = P @ P.T
        for a in np.where(u_has)[0]:
            v = _valid(a); M[a, v] = S[a, v]
        n_merge = 0
        with tqdm(total=n_u, desc="Proposing groups") as pbar:
            while True:
                k = int(np.argmax(M)); a, b = divmod(k, n_u); s = float(M[a, b])
                if not np.isfinite(s) or s < SUGGEST_SIM:
                    break
                if not _feasible(segs[a], segs[b]):
                    M[a, b] = M[b, a] = -np.inf
                    continue
                segs[a] += segs[b]; groups[a] += groups[b]
                E[a] += E[b]; P[a] = _norm(E[a])
                if tm[a] < 0: tm[a] = tm[b]
                OV[a] += OV[b]; OV[:, a] = OV[a]; OV[a, a] = 0
                alive[b] = False; M[b, :] = -np.inf; M[:, b] = -np.inf
                v = _valid(a)
                row = np.full(n_u, -np.inf, dtype=np.float32); row[v] = P[v] @ P[a]
                M[a, :] = row; M[:, a] = row
                n_merge += 1; pbar.update(1)

        prop = [(groups[r], P[r], tm[r]) for r in range(n_u) if alive[r]]
        multi = sorted([g for g in prop if len(g[0]) > 1], key=lambda g: -u_nfr[g[0]].sum())
        solo = [g for g in prop if len(g[0]) == 1]
        print(f"Proposed groups: {len(multi)} (sizes {[len(g[0]) for g in multi[:30]]}) | unassigned IDs: {len(solo)}")

        # ---- 3. CSV (edit the `group` column) ----
        recs = []
        for gi, (mem_rows, gp, gt) in enumerate(multi, start=1):
            for r in mem_rows:
                recs.append({"id": u_ids[r], "group": gi, "team": int(u_team[r]),
                             "sim_to_group": round(float(U_P[r] @ gp), 3) if u_has[r] else None,
                             "first_frame": min(f for f, _ in u_obs[u_ids[r]]),
                             "last_frame": max(f for f, _ in u_obs[u_ids[r]]), "n_det": int(u_nfr[r])})
        for mem_rows, gp, gt in solo:
            r = mem_rows[0]
            recs.append({"id": u_ids[r], "group": -1, "team": int(u_team[r]), "sim_to_group": None,
                         "first_frame": min(f for f, _ in u_obs[u_ids[r]]),
                         "last_frame": max(f for f, _ in u_obs[u_ids[r]]), "n_det": int(u_nfr[r])})
        union_df = pd.DataFrame(recs).sort_values(["group", "sim_to_group", "first_frame"], na_position="last")
        union_df.to_csv(UNION_CSV, index=False)
        print(f"Saved {UNION_CSV}: group -1 = unassigned. Move an ID by writing another group number (or a new one).")

        for g, gdf in union_df[union_df.group > 0].groupby("group"):
            gdf = gdf.sort_values("sim_to_group")
            _sheet(f"GROUP {g} | team {gdf.team.mode().iloc[0]} | {len(gdf)} IDs | {int(gdf.n_det.sum())} det  "
                   f"(least similar first)", gdf.id.tolist(), gdf.sim_to_group.tolist(),
                   os.path.join(UNION_DIR, f"group_{g:03d}.png"), SHOW_GROUPS_INLINE)
        un = union_df[union_df.group < 0].sort_values(["team", "first_frame"])
        for p in range(math.ceil(len(un) / UNASSIGNED_PER_PAGE)):
            part = un.iloc[p * UNASSIGNED_PER_PAGE:(p + 1) * UNASSIGNED_PER_PAGE]
            _sheet(f"UNASSIGNED page {p + 1} | {len(part)} IDs (by team, then time)", part.id.tolist(), [None] * len(part),
                   os.path.join(UNION_DIR, f"unassigned_{p + 1:02d}.png"), False)
        print(f"Sheets saved in {UNION_DIR}/ (group_XXX.png shown above, unassigned_XX.png on disk). "
              f"Edit {UNION_CSV}, upload it back if you edited it offline, then run Cell 19.")


## Cell 5 -- (review) Your decisions on the group sheets
Already filled with the review you did. Safe to re-run.

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # ============================================================================================
    # Cell 18b -- Apply your review of the Cell 18 sheets to union_groups.csv
    # Safe to re-run: merges/moves are idempotent and DROP_BELOW_SIM only touches round-1 members.
    # After round 2 (Cell 18c), put wrong round-2 tiles in REMOVE_IDS (or MOVE_IDS) and re-run 18b -> 18c.
    # ============================================================================================
    MOVE_IDS       = {1660: 4}               # id -> group: the 31:19 tile (id 1660) in group 10 is Lewandowski
    DROP_BELOW_SIM = {17: 0.9}               # group -> its round-1 tiles below this sim go back to unassigned
    MERGE_GROUPS   = {21: 17}                # group 21 is the same player as group 17
    MIXED_GROUPS   = list(range(22, 36))     # mixed / unreviewed groups (22-35) -> every ID back to unassigned (-1)
    REMOVE_IDS     = []                      # ids -> unassigned (use after reviewing round-2 sheets)

    udf = pd.read_csv(UNION_CSV)
    if "round" not in udf.columns:
        udf["round"] = np.where(udf.group > 0, 1, 0)

    for g, th in DROP_BELOW_SIM.items():
        m = (udf.group == g) & (udf["round"] == 1) & (udf.sim_to_group < th)
        print(f"group {g}: {int(m.sum())} tiles below sim {th} -> unassigned: {udf.id[m].tolist()}")
        udf.loc[m, ["group", "round"]] = [-1, 0]
    for g_from, g_to in MERGE_GROUPS.items():
        m = udf.group == g_from
        if m.any():
            print(f"group {g_from} ({int(m.sum())} IDs) merged into group {g_to}")
        udf.loc[m, "group"] = g_to
    for cid, g in MOVE_IDS.items():
        m = udf.id == cid
        if not m.any():
            print(f"  [warn] id {cid} not found -- check the red number on the tile"); continue
        print(f"id {cid}: group {int(udf.group[m].iloc[0])} -> {g}")
        udf.loc[m, "group"] = g
    m = udf.group.isin(MIXED_GROUPS)
    print(f"mixed groups {MIXED_GROUPS}: {int(m.sum())} IDs -> unassigned")
    udf.loc[m, ["group", "round"]] = [-1, 0]
    m = udf.id.isin(REMOVE_IDS)
    udf.loc[m, ["group", "round"]] = [-1, 0]
    if REMOVE_IDS:
        print(f"removed {int(m.sum())} IDs -> unassigned")

    udf.to_csv(UNION_CSV, index=False)
    _conf = udf[udf.group > 0]
    print(f"\nConfirmed groups: {_conf.group.nunique()} | IDs in them: {len(_conf)} "
          f"({int(_conf.n_det.sum())} det) | unassigned IDs: {int((udf.group < 0).sum())} "
          f"({int(udf.n_det[udf.group < 0].sum())} det)")




## Cell 6 -- (review) Round 2: attach unassigned pieces to the confirmed players

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # ============================================================================================
    # Cell 18c -- Round 2: attach unassigned IDs to the CONFIRMED groups, one sheet per group
    # Only the new tiles need checking: each sheet starts with 4 REFERENCE tiles (confirmed members,
    # no score under them), then the proposed additions, least similar first.
    # Hard constraints: same team, never on screen together with the group, physically reachable.
    # No video read: reuses the tiles extracted by Cell 18.
    # ============================================================================================
    ROUND2_SIM    = None      # None = calibrated: must beat 99.5% of DIFFERENT teammates (IDs seen together)
    ROUND2_MARGIN = 0.03      # best group must beat the second-best group by this much (else stays unassigned)
    N_REF_TILES   = 4

    if ROUND2_SIM is None:
        _a, _b = np.where(np.triu(U_OV > 25, 1))                   # on screen together > 1 s = different people
        _ok = u_has[_a] & u_has[_b] & (u_team[_a] == u_team[_b]) & (u_team[_a] >= 0)
        _s = np.einsum("ij,ij->i", U_P[_a[_ok]], U_P[_b[_ok]])
        ROUND2_SIM = float(max(0.6, np.quantile(_s, 0.995))) if len(_s) >= 50 else 0.8
        print(f"Different-teammate ID similarity: median {np.median(_s):.3f}, 99.5% {np.quantile(_s, .995):.3f} "
              f"({len(_s)} pairs) -> ROUND2_SIM = {ROUND2_SIM:.3f}")

    udf = pd.read_csv(UNION_CSV)
    confirmed = sorted(int(g) for g in udf.group.unique() if g > 0)
    g_rows = {g: [u_row[c] for c in udf.id[udf.group == g] if c in u_row] for g in confirmed}
    g_proto = {g: _norm(U_E[rs].sum(axis=0)) for g, rs in g_rows.items()}
    g_team = {}
    for g, rs in g_rows.items():
        t = [int(u_team[r]) for r in rs if u_team[r] >= 0]
        g_team[g] = Counter(t).most_common(1)[0][0] if t else -1
    g_segs = {g: [s for r in rs for s in u_segs[r]] for g, rs in g_rows.items()}

    pairs = []
    for c in udf.id[udf.group < 0]:
        r = u_row.get(int(c))
        if r is None or not u_has[r]:
            continue
        cands = []
        for g in confirmed:
            if u_team[r] >= 0 and g_team[g] >= 0 and u_team[r] != g_team[g]:
                continue
            if U_OV[r, g_rows[g]].sum() > MERGE_OVERLAP_TOL:
                continue
            cands.append((float(U_P[r] @ g_proto[g]), g))
        cands.sort(reverse=True)
        if not cands or cands[0][0] < ROUND2_SIM:
            continue
        if len(cands) > 1 and cands[0][0] - cands[1][0] < ROUND2_MARGIN:
            continue
        pairs.append((cands[0][0], r, cands[0][1]))

    added = defaultdict(list)
    for s, r, g in sorted(pairs, reverse=True):              # most confident first; keep groups co-visibility-free
        if U_OV[r, g_rows[g]].sum() > MERGE_OVERLAP_TOL or not _feasible(u_segs[r], g_segs[g]):
            continue
        g_rows[g].append(r); g_segs[g] += u_segs[r]
        added[g].append((u_ids[r], s))

    for g, lst in added.items():
        for cid, s in lst:
            m = udf.id == cid
            udf.loc[m, ["group", "sim_to_group"]] = [g, round(s, 3)]
            udf.loc[m, "round"] = 2
    udf.to_csv(UNION_CSV, index=False)
    print(f"Round 2: {sum(len(v) for v in added.values())} IDs attached "
          f"({sum(int(u_nfr[u_row[c]]) for v in added.values() for c, _ in v)} det) | "
          f"still unassigned: {int((udf.group < 0).sum())} ({int(udf.n_det[udf.group < 0].sum())} det)")

    for g in confirmed:
        if not added.get(g):
            continue
        conf_ids = udf[(udf.group == g) & (udf["round"] == 1)].sort_values("sim_to_group", ascending=False).id.tolist()
        refs = conf_ids[:N_REF_TILES]
        new = sorted(added[g], key=lambda t: t[1])
        name = GROUP_NAMES.get(g, f"group {g}")
        _sheet(f"ROUND 2 | GROUP {g} {name} | first {len(refs)} = reference | {len(new)} new (least similar first)",
               refs + [c for c, _ in new], [None] * len(refs) + [s for _, s in new],
               os.path.join(UNION_DIR, f"round2_group_{g:03d}.png"), SHOW_GROUPS_INLINE)
        print(f"  group {g:>2} {name:<16} +{len(new)} IDs")




## Cell 7 -- (review) 'Which player is this?' pages for the pieces still unassigned

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # ============================================================================================
    # Cell 18d -- "Which player is this?" sheets for every piece still unassigned
    # One row per unassigned piece (biggest first): the piece's tile, then its top-3 POSSIBLE players
    # (2 reference tiles each, with group number, name and similarity). A player is only offered if the
    # piece is the same team, never on screen together with that player, and physically reachable.
    # "no possible player" = the piece overlaps every candidate in time -> duplicate/false box -> drop.
    # No video read: reuses the tiles from Cell 18.
    # ============================================================================================
    TOP_K         = 3
    REF_PER_CAND  = 2
    ROWS_PER_PAGE = 10
    SHOW_PAGES    = 2                    # pages shown inline (all pages are saved)
    REVIEW_CSV    = os.path.join(UNION_DIR, "unassigned_review.csv")

    udf = pd.read_csv(UNION_CSV)
    if "round" not in udf.columns:
        udf["round"] = np.where(udf.group > 0, 1, 0)
    confirmed = sorted(int(g) for g in udf.group.unique() if g > 0)
    g_rows = {g: [u_row[c] for c in udf.id[udf.group == g] if c in u_row] for g in confirmed}
    g_proto = {g: _norm(U_E[rs].sum(axis=0)) for g, rs in g_rows.items()}
    g_team = {}
    for g, rs in g_rows.items():
        t = [int(u_team[r]) for r in rs if u_team[r] >= 0]
        g_team[g] = Counter(t).most_common(1)[0][0] if t else -1
    g_segs = {g: [s for r in rs for s in u_segs[r]] for g, rs in g_rows.items()}
    g_refs = {}
    for g in confirmed:
        gd = udf[(udf.group == g)].sort_values(["round", "sim_to_group"], ascending=[True, False])
        g_refs[g] = [int(c) for c in gd.id if int(c) in tile][:REF_PER_CAND]

    recs = []
    for cid in udf.id[udf.group == -1]:
        cid = int(cid); r = u_row.get(cid)
        if r is None:
            continue
        cands = []
        for g in confirmed:
            if u_team[r] >= 0 and g_team[g] >= 0 and u_team[r] != g_team[g]:
                continue
            if U_OV[r, g_rows[g]].sum() > MERGE_OVERLAP_TOL or not _feasible(u_segs[r], g_segs[g]):
                continue
            cands.append((float(U_P[r] @ g_proto[g]) if u_has[r] else float("nan"), g))
        cands.sort(key=lambda t: -t[0] if np.isfinite(t[0]) else 1e9)
        recs.append({"id": cid, "n_det": int(u_nfr[r]), "team": int(u_team[r]), "n_possible": len(cands),
                     "suggest": cands[0][1] if cands else 0, "cands": cands[:TOP_K]})
    recs.sort(key=lambda d: -d["n_det"])

    rv = pd.DataFrame([{"id": d["id"], "n_det": d["n_det"], "seconds": round(d["n_det"] / FPS, 1), "team": d["team"],
                        "n_possible": d["n_possible"], "suggest": d["suggest"],
                        **{f"cand{k + 1}": (d["cands"][k][1] if k < len(d["cands"]) else None) for k in range(TOP_K)},
                        **{f"sim{k + 1}": (round(d["cands"][k][0], 3) if k < len(d["cands"]) else None) for k in range(TOP_K)}}
                       for d in recs],
                      columns=["id", "n_det", "seconds", "team", "n_possible", "suggest"]
                              + [f"cand{k + 1}" for k in range(TOP_K)] + [f"sim{k + 1}" for k in range(TOP_K)])
    rv.to_csv(REVIEW_CSV, index=False)
    _tot = int(rv.n_det.sum()) if len(rv) else 0
    print(f"Unassigned pieces: {len(rv)} ({_tot} det) | with no possible player (-> drop): "
          f"{int((rv.n_possible == 0).sum()) if len(rv) else 0}")
    if len(rv):
        for n in (10, 30, 100):
            print(f"  biggest {n:>3} pieces hold {rv.n_det.iloc[:n].sum() / max(_tot, 1):.0%} of the unassigned detections")

    font = cv2.FONT_HERSHEY_SIMPLEX
    LH, GAP, CGAP = 30, 18, 10
    ROW_W = TILE_W + GAP + TOP_K * (REF_PER_CAND * TILE_W + CGAP)

    def _row_img(k, d):
        img = np.full((TILE_H + LH, ROW_W, 3), 255, np.uint8)
        if d["id"] in tile:
            img[:TILE_H, :TILE_W] = tile[d["id"]]
        cv2.rectangle(img, (0, 0), (TILE_W - 1, TILE_H - 1), (0, 0, 200), 2)
        cv2.putText(img, f"#{k} id {d['id']}", (1, TILE_H + 12), font, 0.33, (0, 0, 180), 1, cv2.LINE_AA)
        cv2.putText(img, f"{d['n_det'] / FPS:.0f}s", (1, TILE_H + 25), font, 0.33, (0, 0, 0), 1, cv2.LINE_AA)
        x = TILE_W + GAP
        if not d["cands"]:
            cv2.putText(img, "no possible player -> drop", (x, TILE_H // 2), font, 0.45, (0, 0, 180), 1, cv2.LINE_AA)
        for j, (s, g) in enumerate(d["cands"]):
            for q, rid in enumerate(g_refs.get(g, [])):
                img[:TILE_H, x + q * TILE_W:x + (q + 1) * TILE_W] = tile[rid]
            cv2.rectangle(img, (x, 0), (x + REF_PER_CAND * TILE_W - 1, TILE_H - 1), (150, 150, 150), 1)
            nm = GROUP_NAMES.get(g, f"group {g}")
            cv2.putText(img, f"{j + 1}) G{g} {nm[:13]}", (x + 1, TILE_H + 12), font, 0.33, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(img, f"sim {s:.2f}", (x + 1, TILE_H + 25), font, 0.33, (90, 90, 90), 1, cv2.LINE_AA)
            x += REF_PER_CAND * TILE_W + CGAP
        return img

    for p in range(math.ceil(len(recs) / ROWS_PER_PAGE)):
        chunk = recs[p * ROWS_PER_PAGE:(p + 1) * ROWS_PER_PAGE]
        head = np.full((34, ROW_W, 3), 255, np.uint8)
        cv2.putText(head, f"page {p + 1} | red = piece | 1) = suggested", (4, 22), font, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
        rows_img = [head] + [_row_img(p * ROWS_PER_PAGE + i + 1, d) for i, d in enumerate(chunk)]
        page = np.vstack([np.vstack([r_, np.full((6, ROW_W, 3), 220, np.uint8)]) for r_ in rows_img])
        path = os.path.join(UNION_DIR, f"unassigned_review_{p + 1:02d}.png")
        cv2.imwrite(path, page)
        if p < SHOW_PAGES:
            plt.figure(figsize=(ROW_W / 55, page.shape[0] / 55))
            plt.imshow(cv2.cvtColor(page, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    print(f"Saved {math.ceil(len(recs) / ROWS_PER_PAGE)} pages to {UNION_DIR}/unassigned_review_XX.png and {REVIEW_CSV}")




## Cell 8 -- (review) Your answers from the pages (ACCEPT_SUGGESTED / OVERRIDE)

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # ============================================================================================
    # Cell 18e -- Apply your answers from the review pages
    #   ACCEPT_SUGGESTED : "all", or a list of ids whose suggestion (candidate 1) is right
    #   OVERRIDE         : {id: group}  -> when the right player is candidate 2/3 (or another group)
    #                      {id: 0}      -> drop the piece (not a real player / can't tell)
    # OVERRIDE wins over ACCEPT_SUGGESTED. Rows you don't mention stay unassigned.
    # Typical use: ACCEPT_SUGGESTED = "all" + OVERRIDE for the rows where suggestion 1 is wrong.
    # Cell 19 still refuses anything co-visible, so a slip cannot put two players in one frame.
    # ============================================================================================
    ACCEPT_SUGGESTED = []
    OVERRIDE = {}

    rv = pd.read_csv(REVIEW_CSV)
    udf = pd.read_csv(UNION_CSV)
    if "round" not in udf.columns:
        udf["round"] = np.where(udf.group > 0, 1, 0)
    acc = rv.id.tolist() if ACCEPT_SUGGESTED == "all" else list(ACCEPT_SUGGESTED)
    decide = {int(c): int(rv.suggest[rv.id == c].iloc[0]) for c in acc if (rv.id == c).any()}
    decide.update({int(c): int(g) for c, g in OVERRIDE.items()})
    n_as = n_dr = det_as = det_dr = 0
    for cid, g in decide.items():
        m = udf.id == cid
        if not m.any():
            print(f"  [warn] id {cid} not in the CSV"); continue
        if g > 0:
            udf.loc[m, ["group", "round"]] = [g, 3]; n_as += 1; det_as += int(udf.n_det[m].iloc[0])
        else:
            udf.loc[m, ["group", "round"]] = [-2, 3]; n_dr += 1; det_dr += int(udf.n_det[m].iloc[0])
    udf.to_csv(UNION_CSV, index=False)
    print(f"Assigned {n_as} pieces ({det_as} det) | dropped {n_dr} ({det_dr} det) | "
          f"still unassigned {int((udf.group == -1).sum())} ({int(udf.n_det[udf.group == -1].sum())} det)")
    print("Next: Cell 18c again (optional, groups grew), then Cell 18d for what is left, or Cell 19 to finish.")


## Cell 9 -- (review) Untangle two players mixed between two groups (Simeone 19 / Gimenez 20)

In [ ]:
if MODE != "review":
    print("MODE=apply -> skipped (review step)")
else:
    # ============================================================================================
    # Cell 21 -- Untangle two players whose pieces got mixed between two groups (default: 19 and 20)
    # Key fact: two players on the pitch are on screen TOGETHER most of the time, and two pieces that
    # are on screen together can never be the same player. So the pieces of groups A+B form a
    # "can't-be-the-same" graph that splits them into two sides exactly (2-colouring), connected
    # chunk by connected chunk. Appearance only decides which side is which for each chunk, and alone
    # only for pieces that never overlap anyone. Low-margin decisions stay unassigned.
    # A piece that clashes with BOTH sides is a third person / duplicate box -> unassigned.
    # ============================================================================================
    from collections import deque

    SPLIT_GROUPS  = (19, 20)       # the two mixed groups (Simeone = 19, Gimenez = 20)
    ORIENT_MARGIN = 0.02           # appearance margin needed to orient a chunk / place a lone piece

    udf = pd.read_csv(UNION_CSV)
    sp_ids = [int(c) for c in udf.id[udf.group.isin(SPLIT_GROUPS)] if int(c) in u_row]
    sp_r = np.array([u_row[c] for c in sp_ids])
    n_sp = len(sp_ids)
    w_sp = u_nfr[sp_r].astype(float)
    Adj = U_OV[np.ix_(sp_r, sp_r)] > MERGE_OVERLAP_TOL
    np.fill_diagonal(Adj, False)
    print(f"Pieces in groups {SPLIT_GROUPS}: {n_sp} ({int(w_sp.sum())} det) | 'on screen together' pairs: {int(Adj.sum() // 2)} "
          f"| pieces with at least one such pair: {int(Adj.any(1).sum())}")

    # ---- 1. 2-colouring; a piece closing an odd cycle (clashes with both sides) is removed, smallest first ----
    active = np.ones(n_sp, dtype=bool)
    clash = []
    while True:
        color = -np.ones(n_sp, dtype=int); comp = -np.ones(n_sp, dtype=int); bad = None
        c_id = 0
        for s in np.argsort(-w_sp):
            if not active[s] or comp[s] >= 0:
                continue
            comp[s], color[s] = c_id, 0
            q = deque([s])
            while q and bad is None:
                u = q.popleft()
                for v in np.where(Adj[u] & active)[0]:
                    if comp[v] < 0:
                        comp[v], color[v] = c_id, 1 - color[u]; q.append(v)
                    elif color[v] == color[u]:
                        bad = (u, v); break
            if bad is not None:
                break
            c_id += 1
        if bad is None:
            break
        drop = bad[0] if w_sp[bad[0]] < w_sp[bad[1]] else bad[1]
        active[drop] = False; clash.append(sp_ids[drop])
    print(f"Pieces clashing with both sides (third person / duplicate) -> unassigned: {len(clash)} {clash[:20]}")

    # ---- 2. orient each chunk by appearance, biggest chunk first (it defines side 0 / side 1) ----
    chunks = defaultdict(list)
    for k in range(n_sp):
        if active[k]:
            chunks[comp[k]].append(k)
    order = sorted(chunks, key=lambda c: -w_sp[chunks[c]].sum())
    side = -np.ones(n_sp, dtype=int); margin = np.full(n_sp, np.nan)
    proto_sum = [np.zeros(512, np.float32), np.zeros(512, np.float32)]
    seeded = False
    for c in order:
        ks = chunks[c]
        has_edges = Adj[np.ix_(ks, ks)].any()
        if not seeded:
            if not has_edges:
                continue                                          # need a chunk with both players in it
            for k in ks:
                side[k] = color[k]; proto_sum[color[k]] += U_E[sp_r[k]]
            seeded = True; continue
        P = [_norm(p) for p in proto_sum]
        keep = sum(w_sp[k] * float(U_P[sp_r[k]] @ P[color[k]]) for k in ks if u_has[sp_r[k]])
        swap = sum(w_sp[k] * float(U_P[sp_r[k]] @ P[1 - color[k]]) for k in ks if u_has[sp_r[k]])
        wsum = sum(w_sp[k] for k in ks if u_has[sp_r[k]]) or 1.0
        mg = (keep - swap) / wsum
        for k in ks:
            margin[k] = abs(mg)
        if abs(mg) < ORIENT_MARGIN:
            continue                                              # unsure -> unassigned
        for k in ks:
            side[k] = color[k] if mg > 0 else 1 - color[k]
            proto_sum[side[k]] += U_E[sp_r[k]]
    if not seeded:
        print("  [warn] the two groups are never on screen together -> no constraint, appearance only; not splitting")

    # ---- 3. write back ----
    for k, cid in enumerate(sp_ids):
        g = SPLIT_GROUPS[side[k]] if side[k] >= 0 else -1
        udf.loc[udf.id == cid, ["group", "round"]] = [g, 4]
    udf.to_csv(UNION_CSV, index=False)
    for s_ in (0, 1):
        ks = [k for k in range(n_sp) if side[k] == s_]
        print(f"side {s_} -> group {SPLIT_GROUPS[s_]}: {len(ks)} pieces, {int(w_sp[ks].sum())} det")
    un_k = [k for k in range(n_sp) if side[k] < 0]
    print(f"unassigned (clash or unsure orientation): {len(un_k)} pieces, {int(w_sp[un_k].sum())} det")

    # ---- 4. sheets: check each side is ONE player; tiles in time order, number = orientation margin ----
    P = [_norm(p) for p in proto_sum]
    for s_ in (0, 1):
        ks = sorted([k for k in range(n_sp) if side[k] == s_], key=lambda k: tile_frame.get(sp_ids[k], 0))
        _sheet(f"SPLIT side {s_} -> group {SPLIT_GROUPS[s_]} ({GROUP_NAMES.get(SPLIT_GROUPS[s_], '')}) | {len(ks)} pieces, time order",
               [sp_ids[k] for k in ks], [float(U_P[sp_r[k]] @ P[s_]) if u_has[sp_r[k]] else None for k in ks],
               os.path.join(UNION_DIR, f"split_side{s_}_group{SPLIT_GROUPS[s_]}.png"), True)
    if un_k:
        _sheet(f"SPLIT unassigned ({len(un_k)} pieces)", [sp_ids[k] for k in un_k], [None] * len(un_k),
               os.path.join(UNION_DIR, "split_unassigned.png"), True)
    print("If both sides look like one player each: run Cell 19 (then Cell 20 to name them).")


## Cell 10 -- Apply: final cache with id = team offset + shirt number + name, and the Parquet table

In [ ]:
# Apply the union CSV -> final cache: players get id = team offset + shirt number (+ name, number, team),
# officials keep ids 1-5. Co-visible pieces are never merged (listed instead), so no switch can come from here.
udf = pd.read_csv(UNION_CSV)
udf["group"] = udf["group"].fillna(-1).astype(int)
print(f"Dropped in review (group -2): {int((udf.group == -2).sum())} pieces ({int(udf.n_det[udf.group == -2].sum())} det)")
udf = udf[udf.group != -2]
_unknown_groups = sorted(set(udf.group[udf.group > 0]) - set(PLAYERS))
if _unknown_groups:
    print(f"  [warn] groups without a PLAYERS entry -> treated as unassigned: {_unknown_groups}")
fr_of = {cid: set(f for f, _ in u_obs[cid]) for cid in u_obs}

final_groups, conflicts = [], []                 # (group, [piece ids])
for g, gdf in udf.groupby("group"):
    cids = [int(c) for c in gdf.id if int(c) in fr_of]
    if g < 0 or g not in PLAYERS:
        final_groups += [(-1, [c]) for c in cids]
        continue
    occ, acc = set(), []
    for c in sorted(cids, key=lambda c: -len(fr_of[c])):
        ov = len(fr_of[c] & occ)
        if ov > MERGE_OVERLAP_TOL:
            conflicts.append((g, c, ov)); final_groups.append((-1, [c]))
        else:
            acc.append(c); occ |= fr_of[c]
    final_groups.append((g, acc))

player_info = {}                                        # final id -> dict(name, number, team, class, label)
for tid, (nm, num, tm) in OFFICIALS.items():
    cls = locked_class_by_id.get(tid, "referee" if tid >= 3 else "goalkeeper")
    player_info[tid] = {"name": nm, "number": num, "team": tm, "class": cls,
                 "label": nm if num is None else f"{tm or ''} {num} {nm}".strip()}
remap = {}
next_un = UNASSIGNED_ID_START
for g, cids in sorted(final_groups, key=lambda t: (t[0] < 0, t[0])):
    if g > 0:
        nm, num, tm = PLAYERS[g]
        nid = TEAM_ID_OFFSET[tm] + num
        player_info[nid] = {"name": nm, "number": num, "team": tm, "class": "player", "label": f"{tm} {num} {nm}"}
    else:
        if not KEEP_UNASSIGNED:
            continue
        nid = next_un; next_un += 1
        player_info[nid] = {"name": "unassigned", "number": None, "team": None, "class": "player", "label": f"unassigned {nid}"}
    for c in cids:
        remap[c] = nid

named_tracking_cache, n_dup, table_rows = {}, 0, []
for f in sorted(final_tracking_cache):
    d = final_tracking_cache[f]
    best = {}
    for t in d["tracks"]:
        cls = locked_class_by_id.get(t["track_id"])
        if cls == "player":
            if t["track_id"] not in remap:
                continue                                 # unassigned / dropped piece
            nid = remap[t["track_id"]]
        else:
            nid = t["track_id"]                          # officials keep 1-5
        t2 = dict(t); t2["piece_id"] = t["track_id"]; t2["track_id"] = nid
        t2.update({k: player_info[nid][k] for k in ("name", "number", "team", "label")} if nid in player_info else {})
        b = best.get(nid)
        if b is not None:
            n_dup += 1
        if b is None or t2["conf"] > b["conf"]:
            best[nid] = t2
    named_tracking_cache[f] = {"ball": d["ball"], "tracks": list(best.values())}
    for t2 in best.values():
        x1, y1, x2, y2 = t2["bbox"]
        table_rows.append((f, t2["track_id"], t2.get("name"), t2.get("number"), t2.get("team"),
                           player_info.get(t2["track_id"], {}).get("class"), x1, y1, x2, y2, t2["conf"],
                           t2.get("piece_id"), t2.get("raw_track_id")))
named_locked_class_by_id = {tid: v["class"] for tid, v in player_info.items()}

with open(FINAL_CACHE_PATH, "wb") as fh:
    pickle.dump({"tracking_cache": named_tracking_cache, "locked_class_by_id": named_locked_class_by_id,
                 "player_info": player_info}, fh)
named_df = pd.DataFrame(table_rows, columns=["frame", "track_id", "name", "number", "team", "class",
                                             "x1", "y1", "x2", "y2", "conf", "piece_id", "raw_track_id"])
named_df.to_parquet(FINAL_TABLE_PATH, index=False)

_det = named_df.track_id.value_counts()
print(f"\n{'id':>5}  {'label':<24} {'class':<11} {'det':>7}  {'share':>6}")
for nid in sorted(player_info):
    if nid in _det.index and nid < UNASSIGNED_ID_START:
        print(f"{nid:>5}  {player_info[nid]['label']:<24} {player_info[nid]['class']:<11} {_det[nid]:>7}  {_det[nid] / max(n_frames, 1):>6.1%}")
_un = [n for n in player_info if n >= UNASSIGNED_ID_START]
print(f"\nNamed players: {sum(1 for n in player_info if TEAM_ID_OFFSET['BAR'] <= n < UNASSIGNED_ID_START)} | officials: {len(OFFICIALS)} "
      f"| unassigned kept: {len(_un)} | duplicate boxes resolved: {n_dup}")
print(f"Co-visibility conflicts kept out: {len(conflicts)}")
for g, c, ov in conflicts[:30]:
    print(f"  group {g} ({GROUP_NAMES.get(g, '')}): piece {c} overlaps by {ov} frames -> left out")
print(f"Saved {FINAL_CACHE_PATH}\nSaved {FINAL_TABLE_PATH}")
print("Each track dict now has: track_id (e.g. 102 = BAR #2), name, number, team, label, piece_id (old id), raw_track_id.")


## Cell 10c -- Gap recovery: put deleted boxes back, fill gaps <= 2 s (no new switches)
**A) Recover**: a raw Deep-EIoU run deleted by the tracking notebook (contact piece, short piece, ghost ID) or left unassigned
goes back to player P only if the same raw track is P right before AND right after it, and P has no other box there.
**B) Fill**: remaining gaps of a named ID up to `MAX_FILL_S` are interpolated in pitch space (homography) and reprojected
frame by frame -- only inside one camera shot and only if the move is physically possible (`interpolated=True`, `conf=0`).
**C) Report**: gaps still longer than `MAX_FILL_S`, split into camera cut / left-entered view at the border (normal) /
vanished mid-image (check these). Overwrites the Cell 10 cache and Parquet; the Parquet gets a `source` column
(`tracked` / `recovered` / `interpolated`).

In [ ]:
# ============================================================================================
# Cell 10c -- Gap recovery (notebook 2, run right AFTER Cell 10, before Cell 11)
# Notebook 1 is NOT re-run, so piece ids and your reviewed union CSV stay valid.
# Overwrites the Cell 10 outputs (cache + Parquet) with the recovered version.
#
#  A) RECOVER  every raw Deep-EIoU box that was deleted (contact piece, short piece, ghost ID,
#     unassigned piece) goes back to player P only if the SAME raw track is P right before AND
#     right after the deleted run, and P has no other box at those frames. The tracker followed
#     one box through the run and both ends are P, so this does not create a switch.
#  B) FILL     remaining gaps of a named ID up to MAX_FILL_S are interpolated in PITCH space
#     (homography) and reprojected frame by frame, so camera pans are followed. Only inside one
#     camera shot and only if the move is physically possible. Flagged interpolated=True.
#  C) REPORT   what is still missing. Long gaps are split into: camera cut, player left/entered
#     the view at the image border (normal on broadcast), and "vanished mid-image" (check these).
# ============================================================================================
import os, pickle
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from tqdm.auto import tqdm

# input paths (RAW_TRACKS_PATH, HOMOGRAPHY_PATH, SHOT_CACHE_PATH) are set in Cell 1
MAX_RECOVER_S = 10.0          # longest deleted run given back in step A
MAX_FILL_S = 2.0              # longest gap interpolated in step B (your 1-2 s tolerance)
RESPECT_REVIEW_DROPS = True  # pieces you marked group -2 in the review are never recovered
BORDER_PX = 15                # a box this close to the image edge = player leaving / entering the view
PX_PER_METER = 10
SHOT_HIST_THRESH, SHOT_ADAPTIVE_K = 0.35, 3.0                     # same as notebook 1
CLS_ID_TO_NAME = {1: "player", 2: "referee", 3: "goalkeeper"}

max_rec, max_fill = int(MAX_RECOVER_S * FPS), int(MAX_FILL_S * FPS)


def is_named(nid):
    return nid is not None and nid < UNASSIGNED_ID_START


def new_track(nid, bbox, conf, raw_tid, flag):
    info = player_info.get(nid, {})
    return {"track_id": nid, "bbox": [float(v) for v in bbox], "conf": float(conf),
            "class": info.get("class", "player"), "raw_track_id": raw_tid, "piece_id": None,
            "name": info.get("name"), "number": info.get("number"), "team": info.get("team"),
            "label": info.get("label"), flag: True}


def ownership():
    own, occ = {}, defaultdict(set)
    for f, d in named_tracking_cache.items():
        for t in d["tracks"]:
            occ[t["track_id"]].add(f)
            if t.get("raw_track_id") is not None:
                own[(int(f), int(t["raw_track_id"]))] = t["track_id"]
    return own, occ


raw = pd.read_parquet(RAW_TRACKS_PATH, columns=["frame", "track_id", "x1", "y1", "x2", "y2", "score", "cls"])
raw = raw[raw.cls.isin(list(CLS_ID_TO_NAME))].sort_values(["track_id", "frame"]).reset_index(drop=True)
raw["frame"] = raw.frame.astype(int); raw["track_id"] = raw.track_id.astype(int)


def raw_kept_share():
    own, _ = ownership()
    k = sum(is_named(own.get((f, r))) for f, r in zip(raw.frame.values, raw.track_id.values))
    return k / max(len(raw), 1)


share_before = raw_kept_share()

# ---------------------------------------------------------------- A) recover deleted raw runs
owner, occ = ownership()

# pieces you dropped yourself in the review (group -2) are never given back
blocked = set()
if RESPECT_REVIEW_DROPS and os.path.exists(UNION_CSV):
    _u = pd.read_csv(UNION_CSV)
    _dropped = set(_u.id[_u.group.fillna(-1).astype(int) == -2].astype(int))
    for f, d in final_tracking_cache.items():
        for t in d["tracks"]:
            if t["track_id"] in _dropped and t.get("raw_track_id") is not None:
                blocked.add((int(f), int(t["raw_track_id"])))
    print(f"   review drops respected: {len(_dropped)} pieces ({len(blocked)} boxes) stay out")
add_boxes, remove_keys = defaultdict(list), set()
rec_stats, rej_stats = Counter(), Counter()
for rtid, g in raw.groupby("track_id", sort=False):
    fr = g.frame.to_numpy(); bx = g[["x1", "y1", "x2", "y2"]].to_numpy(float); sc = g.score.to_numpy(float)
    own = [owner.get((int(f), int(rtid))) for f in fr]
    named = np.array([is_named(o) for o in own])
    n, i = len(fr), 0
    while i < n:
        if named[i]:
            i += 1; continue
        j = i
        while j + 1 < n and not named[j + 1]:
            j += 1
        size = j - i + 1
        prev_id = own[i - 1] if i > 0 else None
        next_id = own[j + 1] if j + 1 < n else None
        if prev_id is None or next_id is None:
            rej_stats["one-sided (raw track starts/ends in the run)"] += size
        elif prev_id != next_id:
            rej_stats["different player on each side (ambiguous contact)"] += size
        elif fr[j] - fr[i] + 1 > max_rec:
            rej_stats[f"run longer than {MAX_RECOVER_S:.0f} s"] += size
        elif sum(int(f) in occ[prev_id] for f in fr[i:j + 1]) > MERGE_OVERLAP_TOL:
            rej_stats["player already has another box there"] += size
        else:
            for k in range(i, j + 1):
                f = int(fr[k])
                if f in occ[prev_id]:
                    continue
                if (f, int(rtid)) in blocked:
                    rej_stats["dropped by you in the review (group -2)"] += 1
                    continue
                add_boxes[f].append(new_track(prev_id, bx[k], sc[k], int(rtid), "recovered"))
                occ[prev_id].add(f)
                rec_stats["from unassigned piece" if own[k] is not None else "from deleted box"] += 1
                if own[k] is not None:
                    remove_keys.add((f, int(rtid)))
        i = j + 1

for f, lst in add_boxes.items():
    d = named_tracking_cache.setdefault(f, {"ball": None, "tracks": []})
    d["tracks"] = [t for t in d["tracks"]
                   if t.get("raw_track_id") is None or (f, int(t["raw_track_id"])) not in remove_keys] + lst

print(f"A) recovered boxes: {dict(rec_stats)}")
print(f"   not recovered (raw boxes): {dict(rej_stats)}")

# ---------------------------------------------------------------- B) fill short gaps
with open(HOMOGRAPHY_PATH, "rb") as fh:
    hcache = pickle.load(fh)


def H_at(f):
    return hcache[f] if 0 < f < len(hcache) and hcache[f] is not None else None


def to_pitch(H, x, y):
    return cv2.perspectiveTransform(np.array([[[x, y]]], np.float32), np.asarray(H, np.float64)).reshape(2)


def to_image(H, p):
    return cv2.perspectiveTransform(np.array([[p]], np.float32), np.linalg.inv(np.asarray(H, np.float64))).reshape(2)


SHOT = None
_diff = None
if SHOT_CACHE_PATH and os.path.exists(SHOT_CACHE_PATH):
    _diff = np.load(SHOT_CACHE_PATH)["diff"]
    print(f"   shot cache loaded from {SHOT_CACHE_PATH}")
elif COMPUTE_SHOTS_IF_MISSING:
    # same computation as notebook 1 Cell 7b (one sequential pass over the video, ~5-10 min)
    print("   shot cache not found -> computing camera cuts from the video (same method as notebook 1)")
    _cap = cv2.VideoCapture(VIDEO_PATH)
    assert _cap.isOpened(), f"cannot open {VIDEO_PATH}"
    _diff = np.zeros(n_frames + 1, dtype=np.float32)
    _prev = None
    for _f in tqdm(range(n_frames), desc="Shot detection", unit="frame", mininterval=1.0):
        _ok, _img = _cap.read()
        if not _ok:
            break
        _hsv = cv2.cvtColor(cv2.resize(_img, (160, 90), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2HSV)
        _hist = cv2.calcHist([_hsv], [0, 1], None, [32, 16], [0, 180, 0, 256])
        cv2.normalize(_hist, _hist)
        if _prev is not None:
            _diff[_f] = cv2.compareHist(_prev, _hist, cv2.HISTCMP_BHATTACHARYYA)
        _prev = _hist
    _cap.release()
    np.savez("/kaggle/working/shot_boundaries.npz", diff=_diff)
    print("   saved /kaggle/working/shot_boundaries.npz (add it to a dataset to skip this next time)")
if _diff is not None:
    _med = pd.Series(_diff).rolling(25, center=True, min_periods=1).median().values
    _cut = (_diff > SHOT_HIST_THRESH) & (_diff > SHOT_ADAPTIVE_K * np.maximum(_med, 1e-3)); _cut[0] = False
    SHOT = np.cumsum(_cut)
    print(f"   camera cuts: {int(_cut.sum())}")
else:
    print("   [warn] no shot information -> gaps across camera cuts are not detected")


def same_shot(a, b):
    return SHOT is None or SHOT[min(a, len(SHOT) - 1)] == SHOT[min(b, len(SHOT) - 1)]


_cap = cv2.VideoCapture(VIDEO_PATH)
IMG_W, IMG_H = _cap.get(cv2.CAP_PROP_FRAME_WIDTH), _cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
_cap.release()


def at_border(b):
    return b[0] <= BORDER_PX or b[1] <= BORDER_PX or b[2] >= IMG_W - BORDER_PX or b[3] >= IMG_H - BORDER_PX


by_id = defaultdict(dict)
for f, d in named_tracking_cache.items():
    for t in d["tracks"]:
        if is_named(t["track_id"]):
            by_id[t["track_id"]][f] = t

fill_add, fill_stats, long_gaps = defaultdict(list), Counter(), []
for nid, tf in by_id.items():
    fs = sorted(tf)
    for a, b in zip(fs, fs[1:]):
        gap = b - a - 1
        if gap <= 0:
            continue
        ba, bb = np.asarray(tf[a]["bbox"], float), np.asarray(tf[b]["bbox"], float)
        if gap > max_fill:
            kind = ("camera cut" if not same_shot(a, b) else
                    "left/entered view (border)" if at_border(ba) or at_border(bb) else "vanished mid-image")
            long_gaps.append((nid, a, b, kind)); continue
        if not same_shot(a, b):
            fill_stats["skipped: camera cut in gap"] += gap; continue
        Ha, Hb = H_at(a), H_at(b)
        pa = to_pitch(Ha, (ba[0] + ba[2]) / 2, ba[3]) if Ha is not None else None
        pb = to_pitch(Hb, (bb[0] + bb[2]) / 2, bb[3]) if Hb is not None else None
        if pa is not None and pb is not None and \
                np.linalg.norm(pb - pa) / PX_PER_METER > MAX_SPEED_MPS * (b - a) / FPS + POS_SLACK_M:
            fill_stats["skipped: physically impossible move"] += gap; continue
        for f in range(a + 1, b):
            s = (f - a) / (b - a)
            w, h = (1 - s) * (ba[2:] - ba[:2]) + s * (bb[2:] - bb[:2])
            Hf = H_at(f)
            if pa is not None and pb is not None and Hf is not None:
                fx, fy = to_image(Hf, (1 - s) * pa + s * pb); mode = "pitch"
            else:
                fx = (1 - s) * (ba[0] + ba[2]) / 2 + s * (bb[0] + bb[2]) / 2
                fy = (1 - s) * ba[3] + s * bb[3]; mode = "image"
            if not (0 <= fx < IMG_W and 0 <= fy < IMG_H):
                fill_stats["skipped: off-screen"] += 1; continue
            t = new_track(nid, [fx - w / 2, fy - h, fx + w / 2, fy], 0.0, None, "interpolated")
            t["interp_mode"] = mode
            fill_add[f].append(t); fill_stats[f"filled ({mode} space)"] += 1

for f, lst in fill_add.items():
    named_tracking_cache.setdefault(f, {"ball": None, "tracks": []})["tracks"].extend(lst)
print(f"B) gap filling: {dict(fill_stats)}")

# ---------------------------------------------------------------- C) report
share_after = raw_kept_share()
print(f"\nC) raw Deep-EIoU boxes under a named ID: {share_before:.1%} -> {share_after:.1%}")
lg = pd.DataFrame(long_gaps, columns=["id", "last_seen", "back_at", "kind"])
lg["seconds"] = (lg.back_at - lg.last_seen - 1) / FPS
print(f"Gaps > {MAX_FILL_S:.0f} s left: {len(lg)}")
print(lg.groupby("kind").seconds.agg(["count", "sum", "max"]).round(1).to_string())
sus = lg[lg.kind == "vanished mid-image"].sort_values("seconds", ascending=False)
print(f"\n'vanished mid-image' gaps (check these on the video), longest first:")
for r in sus.head(25).itertuples():
    m, s = divmod(r.last_seen / FPS, 60)
    print(f"  {player_info[r.id]['label']:<24} frames {r.last_seen}-{r.back_at} "
          f"({int(m):02d}:{s:04.1f}, {r.seconds:.1f} s)")

# ---------------------------------------------------------------- save (overwrites Cell 10 outputs)
rows = []
for f in sorted(named_tracking_cache):
    for t in named_tracking_cache[f]["tracks"]:
        x1, y1, x2, y2 = t["bbox"]
        src = "interpolated" if t.get("interpolated") else "recovered" if t.get("recovered") else "tracked"
        rows.append((f, t["track_id"], t.get("name"), t.get("number"), t.get("team"),
                     player_info.get(t["track_id"], {}).get("class"), x1, y1, x2, y2, t["conf"],
                     t.get("piece_id"), t.get("raw_track_id"), src))
named_df = pd.DataFrame(rows, columns=["frame", "track_id", "name", "number", "team", "class", "x1", "y1", "x2",
                                       "y2", "conf", "piece_id", "raw_track_id", "source"])
for c in ("piece_id", "raw_track_id", "number"):
    named_df[c] = pd.to_numeric(named_df[c], errors="coerce").astype("Int64")
with open(FINAL_CACHE_PATH, "wb") as fh:
    pickle.dump({"tracking_cache": named_tracking_cache, "locked_class_by_id": named_locked_class_by_id,
                 "player_info": player_info}, fh)
named_df.to_parquet(FINAL_TABLE_PATH, index=False)
print(f"\nSaved {FINAL_CACHE_PATH} and {FINAL_TABLE_PATH} | rows by source: {named_df.source.value_counts().to_dict()}")

## Cell 10e -- Keep EVERY box (nothing is removed from the output)
Run after Cell 10c. Every raw Deep-EIoU box still missing is put back as **unknown** (unnamed piece -> `10000 + piece id`,
box deleted in notebook 1 -> one id per raw run from `50001`). With `DETECTION_CACHE_PATH` set, every RF-DETR detection the
tracker never output is kept too (`untracked`, ids from `1000000`), and an interpolated box that overlaps a real detection
takes the detection's bbox. The cell **asserts** that no raw box and no detection is missing, then saves.
Unknown/untracked boxes carry no name, so they cannot cause an ID switch. Parquet `source`: tracked / recovered /
interpolated / unknown / untracked.

In [ ]:
# ============================================================================================
# Cell 10e -- Keep EVERY box (run after Cell 10c). Nothing is removed from the output.
#  1) Every raw Deep-EIoU box that is still not in the final cache is put back:
#       - box of a notebook-1 piece you did not name  -> id UNKNOWN_PIECE_ID_START + piece id ("unknown (piece N)")
#       - box deleted in notebook 1 (contact, short, ghost, duplicate) -> one id per raw run ("unknown")
#     Unknown ids are NOT identities: they never get a name, so they cannot create an ID switch.
#  2) (optional) Every RF-DETR detection the tracker never output is kept too:
#       - it overlaps an interpolated box -> the interpolated box takes the real detection bbox (keeps its name)
#       - it overlaps nothing             -> added as "untracked" (unique id, UNTRACKED_ID_START + k)
#  3) Verify: every raw box and every detection has a box in the output, then save.
# ============================================================================================
UNKNOWN_PIECE_ID_START = 10_000
UNKNOWN_RUN_ID_START = 50_000
UNTRACKED_ID_START = 1_000_000
MATCH_IOU = 0.5               # a detection with IoU >= this to an output box is already kept
SNAP_IOU = 0.3                # a detection this close to an interpolated box replaces its bbox


def _iou(a, B):
    if len(B) == 0:
        return np.zeros(0)
    B = np.asarray(B, float)
    ix = np.clip(np.minimum(a[2], B[:, 2]) - np.maximum(a[0], B[:, 0]), 0, None)
    iy = np.clip(np.minimum(a[3], B[:, 3]) - np.maximum(a[1], B[:, 1]), 0, None)
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1]) - inter + 1e-9)


def _add_unknown(uid, cls, label, piece):
    if uid not in player_info:
        player_info[uid] = {"name": "unknown", "number": None, "team": None, "class": cls, "label": label}
        named_locked_class_by_id[uid] = cls


# ---------------------------------------------------------------- 1) raw tracker boxes
piece_of = {}                                          # (frame, raw id) -> notebook-1 piece id
for f, d in final_tracking_cache.items():
    for t in d["tracks"]:
        if t.get("raw_track_id") is not None:
            piece_of[(int(f), int(t["raw_track_id"]))] = int(t["track_id"])
assert max(piece_of.values()) < UNKNOWN_RUN_ID_START - UNKNOWN_PIECE_ID_START, "piece ids too large for the id scheme"


def _present():
    s = set()
    for f, d in named_tracking_cache.items():
        for t in d["tracks"]:
            if t.get("raw_track_id") is not None:
                s.add((int(f), int(t["raw_track_id"])))
    return s


present = _present()
added = Counter()
next_run = UNKNOWN_RUN_ID_START
for rtid, g in raw.groupby("track_id", sort=False):
    rtid = int(rtid)
    prev_was_run = False
    for f, x1, y1, x2, y2, sc, c in zip(g.frame.values, g.x1.values, g.y1.values, g.x2.values, g.y2.values,
                                        g.score.values, g.cls.values):
        f = int(f)
        if (f, rtid) in present:
            prev_was_run = False
            continue
        cls = CLS_ID_TO_NAME[int(c)]
        piece = piece_of.get((f, rtid))
        if piece is not None:
            uid, label, kind = UNKNOWN_PIECE_ID_START + piece, f"unknown (piece {piece})", "unnamed piece"
            prev_was_run = False
        else:
            if not prev_was_run:
                next_run += 1
            uid, label, kind = next_run, "unknown", "deleted in notebook 1"
            prev_was_run = True
        _add_unknown(uid, cls, label, piece)
        named_tracking_cache.setdefault(f, {"ball": None, "tracks": []})["tracks"].append(
            {"track_id": uid, "bbox": [float(x1), float(y1), float(x2), float(y2)], "conf": float(sc), "class": cls,
             "raw_track_id": rtid, "piece_id": piece, "name": "unknown", "number": None, "team": None,
             "label": label, "unknown": True})
        added[kind] += 1
print(f"1) raw tracker boxes put back as unknown: {dict(added)}")

# ---------------------------------------------------------------- 2) detections the tracker never output
n_untracked = n_snapped = 0
if DETECTION_CACHE_PATH and os.path.exists(DETECTION_CACHE_PATH):
    with open(DETECTION_CACHE_PATH, "rb") as fh:
        det_cache = pickle.load(fh)
    _sample = next((ds[0] for ds in det_cache.values() if ds), None)
    print(f"   detection cache: {len(det_cache)} frames | one detection looks like: {_sample}")

    def _det(d):
        bb = d.get("bbox", d.get("xyxy", d.get("box")))
        c = d.get("class", d.get("class_name", d.get("cls")))
        if isinstance(c, (int, np.integer)):
            c = CLS_ID_TO_NAME.get(int(c), str(c))
        return (None if bb is None else np.asarray(bb, float)), str(c), float(d.get("conf", d.get("confidence", 0.0)))

    for f, ds in det_cache.items():
        f = int(f)
        fd = named_tracking_cache.setdefault(f, {"ball": None, "tracks": []})
        for d in ds:
            bb, cls, conf = _det(d)
            if bb is None or cls == "ball":
                continue
            boxes = [t["bbox"] for t in fd["tracks"]]
            ious = _iou(bb, boxes)
            k = int(np.argmax(ious)) if len(ious) else -1
            if k >= 0 and ious[k] >= SNAP_IOU and fd["tracks"][k].get("interpolated") \
                    and not fd["tracks"][k].get("snapped_to_detection"):
                fd["tracks"][k]["bbox"] = bb.tolist(); fd["tracks"][k]["conf"] = conf
                fd["tracks"][k]["snapped_to_detection"] = True       # real detection replaces the estimate
                n_snapped += 1
                continue
            if k >= 0 and ious[k] >= MATCH_IOU:
                continue
            uid = UNTRACKED_ID_START + n_untracked
            n_untracked += 1
            _add_unknown(uid, cls, "untracked", None)
            fd["tracks"].append({"track_id": uid, "bbox": bb.tolist(), "conf": conf, "class": cls,
                                 "raw_track_id": None, "piece_id": None, "name": "unknown", "number": None,
                                 "team": None, "label": "untracked", "unknown": True, "untracked": True})
    print(f"2) detections never output by the tracker: {n_untracked} added as 'untracked', "
          f"{n_snapped} interpolated boxes snapped to a real detection")
else:
    print("2) DETECTION_CACHE_PATH not set -> only raw tracker boxes are guaranteed (set it in Cell 1 to keep "
          "every RF-DETR detection too)")

# ---------------------------------------------------------------- 3) verify nothing is missing
_p = _present()
_miss = sum((int(f), int(r)) not in _p for f, r in zip(raw.frame.values, raw.track_id.values))
print(f"\n3) raw Deep-EIoU boxes missing from the output: {_miss} of {len(raw):,}")
assert _miss == 0, "some raw tracker boxes are still missing"
if DETECTION_CACHE_PATH and os.path.exists(DETECTION_CACHE_PATH):
    _dmiss = 0
    for f, ds in det_cache.items():
        boxes = [t["bbox"] for t in named_tracking_cache.get(int(f), {"tracks": []})["tracks"]]
        for d in ds:
            bb, cls, _ = _det(d)
            if bb is not None and cls != "ball" and not (_iou(bb, boxes) >= MATCH_IOU).any():
                _dmiss += 1
    print(f"   detections without a box in the output: {_dmiss}")
    assert _dmiss == 0, "some detections are still missing"

# ---------------------------------------------------------------- save (overwrites Cell 10/10c outputs)
rows = []
for f in sorted(named_tracking_cache):
    for t in named_tracking_cache[f]["tracks"]:
        x1, y1, x2, y2 = t["bbox"]
        src = ("untracked" if t.get("untracked") else "unknown" if t.get("unknown") else
               "interpolated" if t.get("interpolated") else "recovered" if t.get("recovered") else "tracked")
        rows.append((f, t["track_id"], t.get("name"), t.get("number"), t.get("team"),
                     player_info.get(t["track_id"], {}).get("class", t.get("class")), x1, y1, x2, y2, t["conf"],
                     t.get("piece_id"), t.get("raw_track_id"), src))
named_df = pd.DataFrame(rows, columns=["frame", "track_id", "name", "number", "team", "class", "x1", "y1", "x2",
                                       "y2", "conf", "piece_id", "raw_track_id", "source"])
for c in ("piece_id", "raw_track_id", "number"):
    named_df[c] = pd.to_numeric(named_df[c], errors="coerce").astype("Int64")
with open(FINAL_CACHE_PATH, "wb") as fh:
    pickle.dump({"tracking_cache": named_tracking_cache, "locked_class_by_id": named_locked_class_by_id,
                 "player_info": player_info}, fh)
named_df.to_parquet(FINAL_TABLE_PATH, index=False)
print(f"\nSaved {FINAL_CACHE_PATH} and {FINAL_TABLE_PATH}")
print(f"rows by source: {named_df.source.value_counts().to_dict()}")
print("Analytics: use source in {tracked, recovered, interpolated} for per-player stats; "
      "'unknown' / 'untracked' boxes are real people without a confirmed name.")

## Cell 10f -- Resolve every unknown box into one of the 25 people (no box removed)
Run after Cell 10e. Every unknown / untracked box is grouped into a unit and given the identity that fits under hard rules
(not already on screen in those frames, detector class, team, pitch-speed reachability), scored by appearance + motion
continuity, in passes: confident -> normal -> forced -> relaxed. A box that cannot be any of the 25 is still labelled:
`duplicate` (second box on an already boxed person), `off-pitch` (bench / staff / ball boys), `extra`.
The cell asserts that **no unknown box is left**. Parquet `source`: tracked / recovered / interpolated / auto /
duplicate / off-pitch / extra, plus `assign_score` for auto boxes.

In [ ]:
# ============================================================================================
# Cell 10f -- Resolve EVERY unknown / untracked box (run after Cell 10e). No box is removed.
# Logic: on the pitch there are only 25 people (20 players, 2 GK, 3 referees). Each unknown box is
# grouped into a UNIT (an unnamed piece, a raw run deleted in notebook 1, or a chain of untracked
# detections) and the unit gets the identity that fits, under hard rules:
#   * exclusivity  : the identity is not already on screen in the unit's frames (a person is in one place)
#   * class        : detector class of the unit (player / goalkeeper / referee) when it is clear (>= 70%)
#   * team         : same team as the identity (embedding team label) when both are confident
#   * physics      : reachable from the identity's last box before and next box after (pitch speed limit)
# Score = appearance similarity (Deep-EIoU embeddings) - motion penalty + continuity bonus.
# Passes: "confident" (clear winner) -> "normal" -> "forced" (best feasible) -> "relaxed" (no physics).
# A box that can NOT be any of the 25 is still labelled, never unknown:
#   * duplicate  : a second box on a person already boxed (IoU >= DUP_IOU) -> that person, duplicate=True
#   * off-pitch  : foot position outside the pitch (bench, staff, ball boys)
#   * extra      : all fitting identities are already on screen in those frames
# Interpolated estimates are replaced where a real box is now assigned. Nothing else changes.
# ============================================================================================
OFFPITCH_MARGIN_M = 4.0
DUP_IOU, DUP_SHARE = 0.6, 0.8
ANCHOR_WINDOW_S = 10.0
CHAIN_IOU = 0.4               # untracked detections in consecutive frames with this IoU form one unit
CLASS_GATE_SHARE = 0.7
PASSES = [("confident", 0.08, 0.55), ("normal", 0.03, 0.40), ("forced", None, None), ("relaxed", None, None)]
EXTRA_ID_START = 900_000
EMB_SAMPLE_PER_ID = 3000
PITCH_L_M, PITCH_W_M = 105.0, 68.0

win = int(ANCHOR_WINDOW_S * FPS)
NF = max(named_tracking_cache) + 2
identities = sorted(n for n in player_info if is_named(n))
id_class = {n: player_info[n]["class"] for n in identities}
print(f"Identities: {len(identities)} ({Counter(id_class.values())})")


def _iou1(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def pitch_m(f, bb):
    H = H_at(int(f))
    if H is None:
        return None
    return to_pitch(H, (bb[0] + bb[2]) / 2, bb[3]) / PX_PER_METER


def is_real(t):
    return not (t.get("interpolated") or t.get("duplicate") or t.get("unknown") or t.get("untracked"))


# ---------------------------------------------------------------- 1) units
units, prev_chains, k_ut = {}, [], 0
for f in sorted(named_tracking_cache):
    cur_chains, used = [], set()
    for t in named_tracking_cache[f]["tracks"]:
        if not (t.get("unknown") or t.get("untracked")):
            continue
        if t.get("untracked"):
            key = None
            for ck, cb, cf in prev_chains:
                if cf == f - 1 and ck not in used and _iou1(cb, t["bbox"]) >= CHAIN_IOU:
                    key = ck; break
            if key is None:
                key = ("ut", k_ut); k_ut += 1
            used.add(key); cur_chains.append((key, t["bbox"], f))
        else:
            key = ("uk", t["track_id"])
        u = units.setdefault(key, {"frames": [], "refs": [], "cls": Counter()})
        u["frames"].append(f); u["refs"].append(t); u["cls"][t.get("class", "player")] += 1
    prev_chains = cur_chains
print(f"Units: {len(units)} ({sum(len(u['frames']) for u in units.values()):,} boxes) | "
      f"untracked chains: {sum(1 for k in units if k[0] == 'ut')}")

# ---------------------------------------------------------------- 2) appearance: identities + units
z = np.load(RAW_EMBS_PATH)
emb_idx = {(int(a), int(b)): i for i, (a, b) in enumerate(zip(z["frame"], z["track_id"]))}
EMB = z["emb"]


def mean_emb(keys, cap=None):
    idx = [emb_idx[k] for k in keys if k in emb_idx]
    if not idx:
        return None
    if cap and len(idx) > cap:
        idx = [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]
    return _norm(EMB[idx].astype(np.float32).mean(axis=0))


_id_keys = defaultdict(list)
for f in sorted(named_tracking_cache):
    for t in named_tracking_cache[f]["tracks"]:
        if t["track_id"] in id_class and is_real(t) and t.get("raw_track_id") is not None:
            _id_keys[t["track_id"]].append((int(f), int(t["raw_track_id"])))
proto = {n: mean_emb(_id_keys[n], EMB_SAMPLE_PER_ID) for n in identities}
id_team = {n: (team_label(proto[n]) if proto[n] is not None else -1) for n in identities}
_tm = defaultdict(Counter)
for n in identities:
    if id_class[n] == "player":
        _tm[player_info[n]["team"]][id_team[n]] += 1
print("Team label check (should be one label per team):", {k: dict(v) for k, v in _tm.items()})

for key, u in units.items():
    u["fa"] = np.asarray(u["frames"], dtype=np.int64)
    u["boxes"] = [t["bbox"] for t in u["refs"]]
    u["emb"] = mean_emb([(f, int(t["raw_track_id"])) for f, t in zip(u["frames"], u["refs"])
                         if t.get("raw_track_id") is not None])
    u["team"] = team_label(u["emb"]) if u["emb"] is not None else -1
    c, n = u["cls"].most_common(1)[0]
    u["cls_major"] = c if n >= CLASS_GATE_SHARE * len(u["frames"]) else None

# ---------------------------------------------------------------- 3) state: occupancy + anchors
def build_state():
    occ = {n: np.zeros(NF, dtype=bool) for n in identities}
    fs, bs = defaultdict(list), defaultdict(list)
    for f in sorted(named_tracking_cache):
        for t in named_tracking_cache[f]["tracks"]:
            n = t["track_id"]
            if n in occ and is_real(t):
                occ[n][f] = True; fs[n].append(f); bs[n].append(t["bbox"])
    anc = {n: (np.asarray(fs[n], dtype=np.int64), np.asarray(bs[n], dtype=np.float64).reshape(-1, 4))
           for n in identities}
    return occ, anc


def link(fa, ba, fb, bb):
    """(feasible, motion ratio 0..1+, continuity bonus) between a box at fa and a box at fb > fa."""
    dt = max(int(fb) - int(fa), 1)
    bonus = 0.2 if dt <= 3 and _iou1(ba, bb) >= 0.2 else 0.0
    pa, pb = pitch_m(fa, ba), pitch_m(fb, bb)
    if pa is not None and pb is not None:
        lim = MAX_SPEED_MPS * dt / FPS + POS_SLACK_M
        d = float(np.linalg.norm(pb - pa))
        return d <= lim, d / lim, bonus
    if dt <= 5:
        h = max(ba[3] - ba[1], bb[3] - bb[1], 1.0)
        d = float(np.hypot((ba[0] + ba[2] - bb[0] - bb[2]) / 2, ba[3] - bb[3]))
        lim = h * (0.5 + 0.3 * dt)
        return d <= lim, d / lim, bonus
    return True, 0.5, bonus


def evaluate(u, n, relaxed=False):
    fr = u["fa"]
    if occ[n][fr].sum() > MERGE_OVERLAP_TOL:
        return None
    if u["cls_major"] is not None and u["cls_major"] != id_class[n]:
        return None
    if id_class[n] == "player" and u["team"] >= 0 and id_team[n] >= 0 and u["team"] != id_team[n]:
        return None
    fs, bs = anc[n]
    lo, hi = np.searchsorted(fs, fr[0]), np.searchsorted(fs, fr[-1], side="right")
    if hi - lo > MERGE_OVERLAP_TOL:                       # identity is elsewhere inside the unit's span
        return None
    sim = float(u["emb"] @ proto[n]) if u["emb"] is not None and proto[n] is not None else float("nan")
    rs, bonus = [], 0.0
    if not relaxed:
        if lo > 0 and fr[0] - fs[lo - 1] <= win:
            ok, r, b = link(fs[lo - 1], bs[lo - 1], fr[0], u["boxes"][0])
            if not ok:
                return None
            rs.append(r); bonus += b
        if hi < len(fs) and fs[hi] - fr[-1] <= win:
            ok, r, b = link(fr[-1], u["boxes"][-1], fs[hi], bs[hi])
            if not ok:
                return None
            rs.append(r); bonus += b
    r = max(rs) if rs else 0.5
    return (sim if np.isfinite(sim) else 0.5) - 0.3 * r + bonus, sim


def add_anchor(n, u):
    fs, bs = anc[n]
    idx = np.searchsorted(fs, u["fa"])
    anc[n] = (np.insert(fs, idx, u["fa"]), np.insert(bs, idx, np.asarray(u["boxes"], dtype=np.float64), axis=0))
    occ[n][u["fa"]] = True


# ---------------------------------------------------------------- 4) duplicates and off-pitch people
occ, anc = build_state()
real_by_frame = defaultdict(list)
for f in sorted(named_tracking_cache):
    for t in named_tracking_cache[f]["tracks"]:
        if t["track_id"] in id_class and is_real(t):
            real_by_frame[f].append((t["track_id"], t["bbox"]))

outcome = {}
for key, u in units.items():
    votes = Counter()
    for f, bb in zip(u["frames"], u["boxes"]):
        best = max(((_iou1(bb, b2), n) for n, b2 in real_by_frame.get(f, [])), default=(0.0, None))
        if best[0] >= DUP_IOU:
            votes[best[1]] += 1
    if votes:
        n, c = votes.most_common(1)[0]
        if c >= DUP_SHARE * len(u["frames"]):
            outcome[key] = ("duplicate", n, 1.0, "-"); continue
    if u["cls_major"] in (None, "player"):
        step = max(1, len(u["frames"]) // 20)
        ps = [pitch_m(f, bb) for f, bb in zip(u["frames"][::step], u["boxes"][::step])]
        ps = [p for p in ps if p is not None]
        if ps:
            out = [p[0] < -OFFPITCH_MARGIN_M or p[0] > PITCH_L_M + OFFPITCH_MARGIN_M or
                   p[1] < -OFFPITCH_MARGIN_M or p[1] > PITCH_W_M + OFFPITCH_MARGIN_M for p in ps]
            if np.mean(out) >= 0.8:
                outcome[key] = ("off-pitch", None, 0.0, "-")
print(f"Duplicates: {sum(o[0] == 'duplicate' for o in outcome.values())} units | "
      f"off-pitch: {sum(o[0] == 'off-pitch' for o in outcome.values())} units")

# ---------------------------------------------------------------- 5) assignment passes
remaining = [k for k in units if k not in outcome]
for pname, min_margin, min_score in PASSES:
    occ, anc = build_state()
    relaxed = pname == "relaxed"
    cand = {}
    for k in remaining:
        sc = [(e[0], n) for n in identities for e in [evaluate(units[k], n, relaxed)] if e is not None]
        cand[k] = sorted(sc, reverse=True)
    marg = {k: (c[0][0] - c[1][0] if len(c) > 1 else 1.0) for k, c in cand.items() if c}
    order = sorted(marg, key=lambda k: (-marg[k], -len(units[k]["fa"])))
    n_new = 0
    for k in order:
        u = units[k]
        for s, n in cand[k]:
            strict = min_score is not None
            if strict and (s < min_score or marg[k] < min_margin):
                break
            e = evaluate(u, n, relaxed)                     # re-check against what this pass already assigned
            if e is None:
                if strict:
                    break
                continue
            add_anchor(n, u)
            outcome[k] = ("assigned", n, e[0], pname); n_new += 1
            break
    remaining = [k for k in remaining if k not in outcome]
    print(f"  pass {pname:<9}: {n_new} units assigned ({sum(len(units[k]['fa']) for k, o in outcome.items() if o[3] == pname):,} boxes) "
          f"| left {len(remaining)}")
for k in remaining:
    outcome[k] = ("extra", None, 0.0, "-")

# ---------------------------------------------------------------- 6) apply
next_extra = EXTRA_ID_START
for k, (kind, n, s, pname) in outcome.items():
    u = units[k]
    if kind in ("off-pitch", "extra"):
        nid = next_extra; next_extra += 1
        player_info[nid] = {"name": kind, "number": None, "team": None, "class": "other", "label": kind}
        named_locked_class_by_id[nid] = "other"
        info = player_info[nid]
    else:
        nid, info = n, player_info[n]
    for t in u["refs"]:
        t.pop("unknown", None)
        if t.pop("untracked", None):
            t["was_untracked"] = True
        t.update({"track_id": nid, "name": info["name"], "number": info["number"], "team": info["team"],
                  "label": info["label"], "class": info["class"], "resolved_as": kind})
        if kind == "duplicate":
            t["duplicate"] = True
        elif kind == "assigned":
            t["auto_assigned"] = True; t["assign_score"] = round(float(s), 3); t["assign_pass"] = pname

n_interp_replaced = n_dup_marked = 0
for f, d in named_tracking_cache.items():
    real_ids = Counter(t["track_id"] for t in d["tracks"] if is_real(t))
    keep = []
    for t in d["tracks"]:
        if t.get("interpolated") and real_ids.get(t["track_id"]):
            n_interp_replaced += 1; continue           # an estimate, now replaced by a real box
        keep.append(t)
    best = {}
    for t in keep:
        if not is_real(t) or t["track_id"] not in id_class:
            continue
        b = best.get(t["track_id"])
        if b is None:
            best[t["track_id"]] = t
        else:                                           # two real boxes, same person, same frame -> lower = duplicate
            lo_t = t if t["conf"] < b["conf"] else b
            lo_t["duplicate"] = True; n_dup_marked += 1
            best[t["track_id"]] = b if lo_t is t else t
    d["tracks"] = keep
for nid in [n for n in player_info if n >= UNKNOWN_PIECE_ID_START and not (EXTRA_ID_START <= n < next_extra)]:
    player_info.pop(nid, None); named_locked_class_by_id.pop(nid, None)        # old unknown / untracked ids

# ---------------------------------------------------------------- 7) report + save
res = Counter()
for k, o in outcome.items():
    res[o[0] if o[0] != "assigned" else f"assigned ({o[3]})"] += len(units[k]["fa"])
print(f"\nBoxes by outcome: {dict(res)}")
print(f"Interpolated estimates replaced by real boxes: {n_interp_replaced} | extra same-frame duplicates marked: {n_dup_marked}")
_left = sum(1 for d in named_tracking_cache.values() for t in d["tracks"] if t.get("unknown") or t.get("untracked"))
print(f"Unknown / untracked boxes left: {_left}")
assert _left == 0

auto_s = Counter()
for k, o in outcome.items():
    if o[0] == "assigned":
        auto_s[o[1]] += len(units[k]["fa"])
print("\nAuto-assigned time per identity (check the biggest ones on the Cell 11 sheets):")
for n, c in auto_s.most_common():
    print(f"  {player_info[n]['label']:<24} {c / FPS:>7.1f} s")

rows = []
for f in sorted(named_tracking_cache):
    for t in named_tracking_cache[f]["tracks"]:
        x1, y1, x2, y2 = t["bbox"]
        src = ("duplicate" if t.get("duplicate") else t.get("resolved_as") if t.get("resolved_as") in ("off-pitch", "extra")
               else "auto" if t.get("auto_assigned") else "interpolated" if t.get("interpolated")
               else "recovered" if t.get("recovered") else "tracked")
        rows.append((f, t["track_id"], t.get("name"), t.get("number"), t.get("team"),
                     player_info.get(t["track_id"], {}).get("class", t.get("class")), x1, y1, x2, y2, t["conf"],
                     t.get("piece_id"), t.get("raw_track_id"), src, t.get("assign_score")))
named_df = pd.DataFrame(rows, columns=["frame", "track_id", "name", "number", "team", "class", "x1", "y1", "x2",
                                       "y2", "conf", "piece_id", "raw_track_id", "source", "assign_score"])
for c in ("piece_id", "raw_track_id", "number"):
    named_df[c] = pd.to_numeric(named_df[c], errors="coerce").astype("Int64")
with open(FINAL_CACHE_PATH, "wb") as fh:
    pickle.dump({"tracking_cache": named_tracking_cache, "locked_class_by_id": named_locked_class_by_id,
                 "player_info": player_info}, fh)
named_df.to_parquet(FINAL_TABLE_PATH, index=False)
print(f"\nSaved {FINAL_CACHE_PATH} and {FINAL_TABLE_PATH}")
print(f"rows by source: {named_df.source.value_counts().to_dict()}")

## Cell 11 -- Verify: one contact sheet per final ID (name in the header)
Reads the video (~5-10 min). All crops of a sheet must be the same player.

In [ ]:
# ============================================================================================
# Verify: one contact sheet per final (named) ID
# Each final ID is cut into SEGMENTS (a new segment starts whenever the Deep-EIoU raw id changes,
# the ID disappears for > SEG_GAP_FRAMES, or every SEG_MAX_FRAMES inside a long piece). One crop per segment = one crop per stitched piece,
# so every link/merge made by the relinking is visible. If all crops on a sheet show the same
# player (kit, number, hair, skin, boots), the ID is clean.
# The video is read SEQUENTIALLY (no seeking) so crops are frame-accurate -- ~5-10 min on Kaggle.
# ============================================================================================
import os, math, pickle, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

SHEET_DIR        = "/kaggle/working/named_id_sheets"
MAX_CROPS_PER_ID = 60        # segments above this are evenly subsampled (the header says so)
SEG_GAP_FRAMES   = 50        # 2 s without the ID -> new segment even if the raw id is the same
SEG_MAX_FRAMES   = 750       # also start a new segment every 30 s, so long raw tracks get several crops
CROP_H, CROP_W   = 160, 80
PAD_RATIO        = 0.10      # enlarge each box by 10% before cropping
COLS             = 12
IDS_TO_SHOW      = None      # None = every ID, or e.g. [6, 7, 12]
SHOW_INLINE      = True

VIDEO = globals().get("VIDEO_PATH", "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv")
FPS_  = globals().get("FPS", 25)

# ---- 1. load the final cache (from this session, else from disk) ----
cache, locked = named_tracking_cache, named_locked_class_by_id
_lab = {tid: v["label"] for tid, v in player_info.items()}

obs = defaultdict(list)                       # tid -> [(frame, raw_id, bbox)]
for f, data in cache.items():
    for t in data["tracks"]:
        obs[t["track_id"]].append((f, t.get("raw_track_id"), t["bbox"]))

_named_lim = globals().get("UNKNOWN_PIECE_ID_START", 10**9)        # unknown/untracked boxes: no sheet
ids = sorted(t for t in obs if t < _named_lim) if IDS_TO_SHOW is None else [i for i in IDS_TO_SHOW if i in obs]
if SHOW_INLINE and len(ids) > 60:
    print(f"{len(ids)} IDs: too many to show inline -> saved to disk only (set IDS_TO_SHOW to view some here)")
    SHOW_INLINE = False
order = {"goalkeeper": 0, "referee": 1, "player": 2}
ids.sort(key=lambda t: (order.get(locked.get(t), 3), -len(obs[t])))

# ---- 2. segments -> one representative frame each (largest box in the middle half) ----
wanted = defaultdict(list)                    # frame -> [(tid, k, bbox)]
seg_info = {}                                 # tid -> (list of (start, end, raw_id, rep_frame), n_segments_total)
for tid in ids:
    o = sorted(obs[tid], key=lambda x: x[0])
    segs, cur = [], [o[0]]
    for prev, x in zip(o, o[1:]):
        if x[1] != prev[1] or x[0] - prev[0] > SEG_GAP_FRAMES or x[0] - cur[0][0] >= SEG_MAX_FRAMES:
            segs.append(cur); cur = [x]
        else:
            cur.append(x)
    segs.append(cur)
    n_total = len(segs)
    if n_total > MAX_CROPS_PER_ID:
        keep = sorted(set(np.linspace(0, n_total - 1, MAX_CROPS_PER_ID).round().astype(int).tolist()))
        segs = [segs[k] for k in keep]
    info = []
    for k, s in enumerate(segs):
        mid = s[len(s) // 4: max(len(s) // 4 + 1, 3 * len(s) // 4)]
        rep = max(mid, key=lambda x: x[2][3] - x[2][1])
        wanted[rep[0]].append((tid, k, rep[2]))
        info.append((s[0][0], s[-1][0], s[0][1], rep[0]))
    seg_info[tid] = (info, n_total)

print(f"IDs: {len(ids)} | crops to extract: {sum(len(v[0]) for v in seg_info.values())} "
      f"from {len(wanted)} frames")


def letterbox(img):
    h, w = img.shape[:2]
    s = min(CROP_W / max(w, 1), CROP_H / max(h, 1))
    r = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    out = np.full((CROP_H, CROP_W, 3), 40, np.uint8)
    y0, x0 = (CROP_H - r.shape[0]) // 2, (CROP_W - r.shape[1]) // 2
    out[y0:y0 + r.shape[0], x0:x0 + r.shape[1]] = r
    return out


# ---- 3. read the video sequentially, crop the wanted frames ----
crops = {}
cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f"cannot open {VIDEO}"
last = max(wanted)
f = 0
with tqdm(total=last + 1, desc="Reading video", unit="frame", mininterval=1.0) as pbar:
    while f <= last:
        if f in wanted:
            ok, img = cap.read()
            if not ok:
                break
            H, W = img.shape[:2]
            for tid, k, (x1, y1, x2, y2) in wanted[f]:
                px, py = (x2 - x1) * PAD_RATIO, (y2 - y1) * PAD_RATIO
                a, b = int(max(0, x1 - px)), int(max(0, y1 - py))
                c, d = int(min(W, x2 + px)), int(min(H, y2 + py))
                if c > a and d > b:
                    crops[(tid, k)] = letterbox(img[b:d, a:c])
            pbar.set_postfix(crops=len(crops), refresh=False)
        else:
            if not cap.grab():
                break
        f += 1
        pbar.update(1)
cap.release()
print(f"Extracted {len(crops)} crops (stopped at frame {f})")

# ---- 4. build one sheet per ID ----
os.makedirs(SHEET_DIR, exist_ok=True)
LABEL_H, HEAD_H = 30, 34
font = cv2.FONT_HERSHEY_SIMPLEX
for tid in tqdm(ids, desc="Building sheets"):
    info, n_total = seg_info[tid]
    n = len(info)
    rows = math.ceil(n / COLS)
    sheet = np.full((HEAD_H + rows * (CROP_H + LABEL_H), COLS * CROP_W, 3), 255, np.uint8)
    head = (f"ID {tid} {_lab.get(tid, '')} | {locked.get(tid, '?')} | {len(obs[tid])} det | {n_total} segments"
            + (f" (showing {n})" if n < n_total else ""))
    cv2.putText(sheet, head, (6, 23), font, 0.6, (0, 0, 0), 2, cv2.LINE_AA)
    for k, (s0, s1, raw, rep) in enumerate(info):
        r, cc = divmod(k, COLS)
        y, x = HEAD_H + r * (CROP_H + LABEL_H), cc * CROP_W
        crop = crops.get((tid, k))
        if crop is not None:
            sheet[y:y + CROP_H, x:x + CROP_W] = crop
        cv2.rectangle(sheet, (x, y), (x + CROP_W - 1, y + CROP_H - 1), (200, 200, 200), 1)
        t = rep / FPS_
        cv2.putText(sheet, f"{int(t // 60):02d}:{int(t % 60):02d} f{rep}", (x + 2, y + CROP_H + 12),
                    font, 0.3, (0, 0, 0), 1, cv2.LINE_AA)
        cv2.putText(sheet, f"raw {raw}", (x + 2, y + CROP_H + 25), font, 0.3, (90, 90, 90), 1, cv2.LINE_AA)
    cv2.imwrite(os.path.join(SHEET_DIR, f"id_{tid:04d}_{_lab.get(tid, locked.get(tid, 'x')).replace(' ', '_')}.png"), sheet)
    if SHOW_INLINE:
        plt.figure(figsize=(COLS * 0.95, 0.4 + rows * 2.0))
        plt.imshow(cv2.cvtColor(sheet, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

shutil.make_archive(SHEET_DIR, "zip", SHEET_DIR)
print(f"Saved {len(ids)} sheets to {SHEET_DIR}/ and {SHEET_DIR}.zip")


## Cell 12 -- Clear crops of chosen players (confirm numbers, e.g. De Paul)
Set `UNKNOWN_IDS = [205]` etc.

In [ ]:
# ============================================================================================
# Clear crops of chosen players (to read / confirm shirt numbers)
# For each unknown ID: the BIGGEST boxes (closest to the camera = most readable), only when the
# player is ISOLATED (no other box touching -> no occlusion), spread over the match (>= MIN_SEP_S
# apart, so you get front AND back views). Full-resolution crops, upscaled + sharpened.
# Reads the video sequentially (frame-accurate) up to the last needed frame.
# ============================================================================================
import os, math, cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

N_CROPS     = 12          # crops per player
MIN_SEP_S   = 20          # seconds between chosen crops
ISO_EXPAND  = 0.15        # a box touching this enlarged box = not isolated
PAD         = 0.12        # padding around the crop
OUT_H       = 320         # display height of each crop (upscaled)
COLS        = 6
CROPS_DIR   = "/kaggle/working/player_crops"
_names = {tid: v["label"] for tid, v in player_info.items()}
UNKNOWN_IDS = []              # final ids to show, e.g. [205, 222] (De Paul, Simeone) -- empty = skip this cell
_cache = named_tracking_cache

if not UNKNOWN_IDS:
    print('UNKNOWN_IDS is empty -> nothing to show')
else:
    def _touch(a, b, e):
        wa, ha = (a[2] - a[0]) * e, (a[3] - a[1]) * e
        return not (b[2] < a[0] - wa or b[0] > a[2] + wa or b[3] < a[1] - ha or b[1] > a[3] + ha)

    cands = defaultdict(list)                       # id -> [(height, frame, bbox)]
    _want = set(UNKNOWN_IDS)
    for f, d in _cache.items():
        tr = d["tracks"]
        for t in tr:
            if t["track_id"] in _want:
                b = t["bbox"]
                if all(not _touch(b, o["bbox"], ISO_EXPAND) for o in tr if o is not t):
                    cands[t["track_id"]].append((b[3] - b[1], f, b))

    picked = {}
    for tid in UNKNOWN_IDS:
        chosen = []
        for h, f, b in sorted(cands[tid], key=lambda x: -x[0]):
            if all(abs(f - g) >= MIN_SEP_S * FPS for _, g, _ in chosen):
                chosen.append((h, f, b))
            if len(chosen) == N_CROPS:
                break
        picked[tid] = sorted(chosen, key=lambda x: x[1])
        print(f"ID {tid} {_names.get(tid, '')}: {len(cands[tid])} isolated boxes, "
              f"picked {len(chosen)} (box height {min((c[0] for c in chosen), default=0):.0f}-"
              f"{max((c[0] for c in chosen), default=0):.0f} px)")

    need = defaultdict(list)
    for tid, lst in picked.items():
        for k, (h, f, b) in enumerate(lst):
            need[f].append((tid, k, b))
    crops_hr = {}
    cap = cv2.VideoCapture(VIDEO_PATH)
    assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
    last, f = max(need), 0
    with tqdm(total=last + 1, desc="Reading video", unit="frame", mininterval=1.0) as pbar:
        while f <= last:
            if f in need:
                ok, img = cap.read()
                if not ok: break
                H, W = img.shape[:2]
                for tid, k, (x1, y1, x2, y2) in need[f]:
                    px, py = (x2 - x1) * PAD, (y2 - y1) * PAD
                    a, b_ = int(max(0, x1 - px)), int(max(0, y1 - py))
                    c, d_ = int(min(W, x2 + px)), int(min(H, y2 + py))
                    cr = img[b_:d_, a:c]
                    if cr.size:
                        s = OUT_H / cr.shape[0]
                        up = cv2.resize(cr, (max(1, int(cr.shape[1] * s)), OUT_H), interpolation=cv2.INTER_CUBIC)
                        blur = cv2.GaussianBlur(up, (0, 0), 2.0)
                        crops_hr[(tid, k)] = cv2.addWeighted(up, 1.6, blur, -0.6, 0)       # unsharp mask
            elif not cap.grab():
                break
            f += 1; pbar.update(1)
    cap.release()

    os.makedirs(CROPS_DIR, exist_ok=True)
    for tid in UNKNOWN_IDS:
        lst = picked[tid]
        if not lst:
            continue
        rows = math.ceil(len(lst) / COLS)
        fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 2.2, rows * 4.0))
        axes = np.atleast_1d(axes).ravel()
        for ax in axes:
            ax.axis("off")
        for k, (h, f, b) in enumerate(lst):
            im = crops_hr.get((tid, k))
            if im is None:
                continue
            axes[k].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            t = f / FPS
            axes[k].set_title(f"{int(t // 60):02d}:{int(t % 60):02d}  f{f}", fontsize=8)
            cv2.imwrite(os.path.join(CROPS_DIR, f"id{tid}_{k:02d}_f{f}.png"), im)
        fig.suptitle(f"ID {tid} -- {_names.get(tid, '')}", fontsize=13, fontweight="bold")
        plt.tight_layout(); plt.show()
    print(f"Full-resolution crops saved in {CROPS_DIR}/")
